
# MVP — Engenharia de Dados

## Pipeline de Dados para Análise do Desempenho Musical no Spotify

**Nome:** Lívia Brandão  
**Matrícula:** 4052025002389  
**Sprint:** Engenharia de Dados  
**Plataforma:** Databricks Free Edition

## Objetivo

Este trabalho tem como objetivo construir um pipeline de dados em nuvem para organizar e analisar informações sobre o desempenho de músicas nas plataformas digitais.

A proposta é partir dos dados em seu formato original e percorrer as etapas de ingestão, armazenamento, tratamento e modelagem até chegar a conjuntos de dados confiáveis e adequados para análise. Para isso, será utilizada a arquitetura medalhão, estruturada nas camadas Bronze, Silver e Gold.

Além do pipeline principal, será avaliada a possibilidade de incorporar uma segunda fonte de dados com avaliações de crítica e de usuários. A intenção é explorar uma dimensão diferente do sucesso musical, comparando alcance nas plataformas digitais com recepção crítica.

## Perguntas do projeto

A construção do pipeline será orientada pelas seguintes perguntas:

1. Quais músicas e artistas estão associados aos maiores volumes de streams?
2. Músicas presentes em mais playlists e charts apresentam também maior número de streams?
3. Como o desempenho das músicas varia de acordo com o período de lançamento?
4. Características musicais como danceability, energy e valence apresentam diferenças entre músicas de maior e menor alcance?
5. O consumo está concentrado em uma pequena parcela das músicas e artistas da base?
6. Existe associação entre o desempenho das músicas nas plataformas digitais e sua recepção pela crítica e pelos usuários?

As perguntas serão mantidas ao longo do projeto para que, ao final, seja possível comparar o que se pretendia investigar com aquilo que os dados efetivamente permitiram responder.

# Fonte, Coleta e Linhagem dos Dados

## Fonte principal

O conjunto de dados principal utilizado é o **Most Streamed Spotify Songs 2023**, disponibilizado publicamente na plataforma Kaggle por Nidula Elgiriyewithana.

A base reúne informações sobre músicas que tiveram destaque nas plataformas digitais. Entre os atributos disponíveis estão nome da faixa, artistas, data de lançamento, número de streams, presença em playlists e charts e características musicais como BPM, danceability, energy e valence.

**Dataset:** Most Streamed Spotify Songs 2023  
**Fonte:** Kaggle  
**Autor:** Nidula Elgiriyewithana  
**DOI:** 10.34740/KAGGLE/DSV/6367938  
**Formato utilizado:** CSV  
**Arquivo utilizado:** spotify-2023.csv  
**Frequência de atualização indicada:** anual

## Proveniência

De acordo com as informações apresentadas no Kaggle, o conjunto foi construído a partir de **múltiplas fontes de dados**, utilizando **Python** como metodologia de coleta. A página do dataset não apresenta citações adicionais que permitam identificar individualmente essas fontes.

Por esse motivo, neste trabalho a origem dos dados é registrada conforme a documentação disponibilizada pelo responsável pelo dataset, sem assumir que os registros tenham sido obtidos diretamente da API oficial do Spotify.

## Licença

No Kaggle, a licença do conjunto de dados está identificada como **“Other (specified in description)”**. Não há indicação, na página consultada, de uma licença padronizada específica, como Creative Commons ou ODbL. Dessa forma, foi mantida neste trabalho a classificação informada pela própria fonte.

## Persistência e processamento

Para tornar o pipeline reproduzível, uma cópia do arquivo CSV utilizado foi armazenada em um repositório público no GitHub. O processamento e a persistência das diferentes camadas do pipeline serão realizados no Databricks Free Edition.

**Origem documentada:** Kaggle  
**Persistência para reprodução:** GitHub público  
**Processamento e armazenamento:** Databricks Free Edition

## Arquitetura e linhagem

O fluxo principal dos dados foi definido da seguinte forma:

**Kaggle → CSV → GitHub → Bronze → Silver → Gold → Análises**

Na camada **Bronze**, os dados são preservados conforme recebidos da fonte, com a inclusão apenas dos metadados técnicos necessários para registrar e rastrear a ingestão. Essa preservação é importante porque permite identificar problemas existentes no arquivo original antes de qualquer tratamento.

Na camada **Silver**, serão realizadas as etapas de limpeza e preparação, incluindo conversão de tipos, tratamento de valores ausentes, padronização, verificação de duplicidades e tratamento de registros que apresentem problemas de estrutura ou conteúdo identificados durante a análise de qualidade da Bronze.

Na camada **Gold**, os dados serão organizados de acordo com as necessidades das análises. Nessa etapa serão estruturadas informações relacionadas a músicas, artistas, tempo e desempenho. Como uma música pode estar associada a mais de um artista, essa relação também será considerada durante a modelagem.

## Enriquecimento dos dados

Como extensão do pipeline principal, será avaliada a inclusão de uma segunda fonte contendo avaliações de crítica e de usuários sobre produções musicais.

A intenção é acrescentar uma dimensão de recepção ao conjunto principal. Enquanto os dados do Spotify permitem observar alcance, exposição e características das músicas, uma fonte de avaliações pode permitir investigar se esse desempenho também apresenta alguma relação com a percepção da crítica e do público.

A fonte candidata inicial é o **MusicBrainz**, que reúne avaliações de críticos e usuários. Antes de sua utilização serão verificadas a disponibilidade dos dados, as condições de uso e a possibilidade de integração consistente com a base principal.

Caso essa integração seja viável, os dados tratados serão incorporados a uma camada Gold enriquecida e utilizados para responder à sexta pergunta do projeto.


# Camada Bronze — Ingestão dos Dados Brutos

A primeira etapa do pipeline consiste na ingestão e persistência dos dados na camada Bronze.

O arquivo `spotify-2023.csv`, originalmente obtido no Kaggle e mantido no GitHub para garantir a reprodutibilidade do projeto, foi coletado via HTTPS e carregado no Databricks.

Nesta camada, os dados são mantidos com sua estrutura original, sem correções, conversões de tipos ou tratamentos de qualidade. Foram acrescentados apenas três metadados técnicos para preservar a rastreabilidade da ingestão:

- `_data_ingestao`: data e hora da ingestão;
- `_arquivo_origem`: identificação do arquivo utilizado;
- `_url_origem`: endereço utilizado para coleta.

A ingestão resultou em **953 registros**, **24 atributos provenientes da fonte** e **3 metadados técnicos**, totalizando **27 colunas**.

Os dados foram persistidos em formato **Delta** na tabela `spotify_bronze`. Os nomes originais dos atributos foram preservados nesta etapa, inclusive `artist(s)_name`.

## Verificação inicial de qualidade

Antes da transformação para a camada Silver, os dados da Bronze foram analisados para identificar problemas que deveriam ser tratados nas etapas seguintes.

Foram verificados:

- valores ausentes;
- registros duplicados;
- tipos de dados;
- valores percentuais fora do intervalo de 0 a 100;
- valores quantitativos negativos;
- intervalos básicos de mês e dia;
- campos numéricos recebidos como texto;
- possibilidade de conversão desses campos para tipos numéricos;
- existência de registros com conteúdo incompatível com o tipo esperado para o atributo.

A análise identificou **95 valores ausentes em `key` (9,97%)** e **50 em `in_shazam_charts` (5,25%)**. Não foram encontrados registros duplicados, percentuais fora do domínio analisado, valores negativos nos campos quantitativos verificados ou meses e dias fora dos intervalos básicos esperados.

Também foi observado que `streams`, `in_deezer_playlists` e `in_shazam_charts` foram recebidos como texto. Alguns registros utilizam vírgula como separador de milhar, o que exige tratamento antes da conversão desses atributos para tipos numéricos.

Durante o aprofundamento da validação de `streams`, foi identificado pelo menos um registro cujo conteúdo não corresponde ao valor numérico esperado para esse atributo. O registro apresenta conteúdo de outros atributos musicais concatenado no campo `streams`, indicando um problema de estrutura ou conteúdo existente no dado de origem.

Esse registro será analisado antes da construção da camada Silver para que o tratamento adotado seja baseado na evidência encontrada, evitando a correção ou exclusão automática de dados sem justificativa.

Os resultados desta análise serão utilizados para definir e documentar as transformações realizadas na camada Silver.

In [0]:
# ============================================================
# CAMADA BRONZE — INGESTÃO DOS DADOS BRUTOS
# ============================================================

import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# Arquivo público utilizado para reprodução do pipeline
url_spotify = "https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv"

# Coleta do CSV via HTTPS
# O encoding ISO-8859-1 é utilizado para preservar os caracteres do arquivo original
df_pandas = pd.read_csv(
    url_spotify,
    encoding="ISO-8859-1"
)

# Conversão para DataFrame Spark
df_bronze = spark.createDataFrame(df_pandas)

# Inclusão somente de metadados técnicos de ingestão
# Os dados originais ainda não são tratados nesta camada
df_bronze = (
    df_bronze
    .withColumn("_data_ingestao", current_timestamp())
    .withColumn("_arquivo_origem", lit("spotify-2023.csv"))
    .withColumn("_url_origem", lit(url_spotify))
)

# Validação inicial da ingestão
print("Quantidade de registros:", df_bronze.count())
print("Quantidade de colunas:", len(df_bronze.columns))

# Visualização dos primeiros registros
display(df_bronze.limit(10))

Quantidade de registros: 953
Quantidade de colunas: 27


track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,in_apple_charts,in_deezer_playlists,in_deezer_charts,in_shazam_charts,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%,_data_ingestao,_arquivo_origem,_url_origem
Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,263,45,10,826,125,B,Major,80,89,83,31,0,8,4,2026-09-27T18:31:07.772Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,126,58,14,382,92,C#,Major,71,61,74,7,0,10,4,2026-09-27T18:31:07.772Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,207,91,14,949,138,F,Major,51,32,53,17,0,31,6,2026-09-27T18:31:07.772Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,207,125,12,548,170,A,Major,55,58,72,11,0,11,15,2026-09-27T18:31:07.772Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,133,87,15,425,144,A,Minor,65,23,80,14,63,11,6,2026-09-27T18:31:07.772Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
Sprinter,"Dave, Central Cee",2,2023,6,1,2186,91,183706234,67,213,88,17,946,141,C#,Major,92,66,58,19,0,8,24,2026-09-27T18:31:07.772Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
Ella Baila Sola,"Eslabon Armado, Peso Pluma",2,2023,3,16,3090,50,725980112,34,222,43,13,418,148,F,Minor,67,83,76,48,0,8,3,2026-09-27T18:31:07.772Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
Columbia,Quevedo,1,2023,7,7,714,43,58149378,25,89,30,13,194,100,F,Major,67,26,71,37,0,11,4,2026-09-27T18:31:07.772Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
fukumean,Gunna,1,2023,5,15,1096,83,95217315,60,210,48,11,953,130,C#,Minor,85,22,62,12,0,28,9,2026-09-27T18:31:07.772Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
La Bebe - Remix,"Peso Pluma, Yng Lvcas",2,2023,3,17,2953,44,553634067,49,110,66,13,339,170,D,Minor,81,56,48,21,0,8,33,2026-09-27T18:31:07.772Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv


In [0]:
# VALIDAÇÃO DA INGESTÃO BRONZE

print("SCHEMA RECEBIDO:")
df_bronze.printSchema()

print("\nCOLUNAS RECEBIDAS:")
for i, coluna in enumerate(df_bronze.columns, start=1):
    print(f"{i:02d}. {coluna}")

print("\nRESUMO DA INGESTÃO:")
print("Registros:", df_bronze.count())
print("Colunas originais:", len(df_bronze.columns) - 3)
print("Metadados técnicos adicionados:", 3)
print("Total de colunas na Bronze:", len(df_bronze.columns))

SCHEMA RECEBIDO:
root
 |-- track_name: string (nullable = true)
 |-- artist(s)_name: string (nullable = true)
 |-- artist_count: long (nullable = true)
 |-- released_year: long (nullable = true)
 |-- released_month: long (nullable = true)
 |-- released_day: long (nullable = true)
 |-- in_spotify_playlists: long (nullable = true)
 |-- in_spotify_charts: long (nullable = true)
 |-- streams: string (nullable = true)
 |-- in_apple_playlists: long (nullable = true)
 |-- in_apple_charts: long (nullable = true)
 |-- in_deezer_playlists: string (nullable = true)
 |-- in_deezer_charts: long (nullable = true)
 |-- in_shazam_charts: string (nullable = true)
 |-- bpm: long (nullable = true)
 |-- key: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- danceability_%: long (nullable = true)
 |-- valence_%: long (nullable = true)
 |-- energy_%: long (nullable = true)
 |-- acousticness_%: long (nullable = true)
 |-- instrumentalness_%: long (nullable = true)
 |-- liveness_%: long (nulla

In [0]:
# ============================================================
# PERSISTÊNCIA DA CAMADA BRONZE
# Preserva os nomes originais das colunas com Delta Column Mapping
# ============================================================

tabela_bronze = "spotify_bronze"

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .saveAsTable(tabela_bronze)
)

# Validação da tabela persistida
df_bronze_salva = spark.table(tabela_bronze)

print("Tabela criada:", tabela_bronze)
print("Registros persistidos:", df_bronze_salva.count())
print("Colunas persistidas:", len(df_bronze_salva.columns))

display(df_bronze_salva.limit(10))

Tabela criada: spotify_bronze
Registros persistidos: 953
Colunas persistidas: 27


track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,in_apple_charts,in_deezer_playlists,in_deezer_charts,in_shazam_charts,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%,_data_ingestao,_arquivo_origem,_url_origem
Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,263,45,10,826,125,B,Major,80,89,83,31,0,8,4,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,126,58,14,382,92,C#,Major,71,61,74,7,0,10,4,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,207,91,14,949,138,F,Major,51,32,53,17,0,31,6,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,207,125,12,548,170,A,Major,55,58,72,11,0,11,15,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,133,87,15,425,144,A,Minor,65,23,80,14,63,11,6,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
Sprinter,"Dave, Central Cee",2,2023,6,1,2186,91,183706234,67,213,88,17,946,141,C#,Major,92,66,58,19,0,8,24,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
Ella Baila Sola,"Eslabon Armado, Peso Pluma",2,2023,3,16,3090,50,725980112,34,222,43,13,418,148,F,Minor,67,83,76,48,0,8,3,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
Columbia,Quevedo,1,2023,7,7,714,43,58149378,25,89,30,13,194,100,F,Major,67,26,71,37,0,11,4,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
fukumean,Gunna,1,2023,5,15,1096,83,95217315,60,210,48,11,953,130,C#,Minor,85,22,62,12,0,28,9,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv
La Bebe - Remix,"Peso Pluma, Yng Lvcas",2,2023,3,17,2953,44,553634067,49,110,66,13,339,170,D,Minor,81,56,48,21,0,8,33,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv


In [0]:
# ============================================================
# DIAGNÓSTICO DE QUALIDADE — CAMADA BRONZE
# ============================================================

from pyspark.sql import functions as F

df = spark.table("spotify_bronze")

print("=== 1. DIMENSÕES DA BASE ===")
print("Registros:", df.count())
print("Colunas:", len(df.columns))

print("\n=== 2. VALORES NULOS POR COLUNA ===")

expressoes_nulos = [
    F.sum(
        F.when(F.col(f"`{c}`").isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
]

display(df.select(expressoes_nulos))

print("\n=== 3. REGISTROS DUPLICADOS ===")

total = df.count()
total_distintos = df.drop(
    "_data_ingestao",
    "_arquivo_origem",
    "_url_origem"
).distinct().count()

print("Total de registros:", total)
print("Registros distintos:", total_distintos)
print("Duplicados:", total - total_distintos)

print("\n=== 4. CAMPOS QUE CHEGARAM COMO STRING ===")

for campo, tipo in df.dtypes:
    if tipo == "string":
        print(f"{campo}: {tipo}")

print("\n=== 5. ESTATÍSTICAS DOS CAMPOS NUMÉRICOS ===")

=== 1. DIMENSÕES DA BASE ===
Registros: 953
Colunas: 27

=== 2. VALORES NULOS POR COLUNA ===


track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,in_apple_charts,in_deezer_playlists,in_deezer_charts,in_shazam_charts,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%,_data_ingestao,_arquivo_origem,_url_origem
0,0,0,0,0,0,0,0,0,0,0,0,0,50,0,95,0,0,0,0,0,0,0,0,0,0,0



=== 3. REGISTROS DUPLICADOS ===
Total de registros: 953
Registros distintos: 953
Duplicados: 0

=== 4. CAMPOS QUE CHEGARAM COMO STRING ===
track_name: string
artist(s)_name: string
streams: string
in_deezer_playlists: string
in_shazam_charts: string
key: string
mode: string
_arquivo_origem: string
_url_origem: string

=== 5. ESTATÍSTICAS DOS CAMPOS NUMÉRICOS ===


In [0]:
# ============================================================
# QUALIDADE — NULOS POR ATRIBUTO
# ============================================================

resultado_nulos = []

for coluna in df.columns:
    qtd_nulos = df.filter(F.col(f"`{coluna}`").isNull()).count()

    resultado_nulos.append(
        (coluna, qtd_nulos, round((qtd_nulos / df.count()) * 100, 2))
    )

df_nulos = spark.createDataFrame(
    resultado_nulos,
    ["atributo", "qtd_nulos", "percentual_nulos"]
)

display(df_nulos.orderBy(F.desc("qtd_nulos")))

atributo,qtd_nulos,percentual_nulos
key,95,9.97
in_shazam_charts,50,5.25
track_name,0,0.0
artist(s)_name,0,0.0
artist_count,0,0.0
released_year,0,0.0
released_month,0,0.0
released_day,0,0.0
in_spotify_playlists,0,0.0
in_spotify_charts,0,0.0


In [0]:
# ============================================================
# QUALIDADE — VALIDAÇÃO DE DOMÍNIOS
# ============================================================

campos_percentuais = [
    "danceability_%",
    "valence_%",
    "energy_%",
    "acousticness_%",
    "instrumentalness_%",
    "liveness_%",
    "speechiness_%"
]

print("=== PERCENTUAIS FORA DO INTERVALO 0–100 ===")

for campo in campos_percentuais:
    invalidos = df.filter(
        (F.col(f"`{campo}`") < 0) |
        (F.col(f"`{campo}`") > 100)
    ).count()

    print(f"{campo}: {invalidos}")

print("\n=== DATAS POTENCIALMENTE INVÁLIDAS ===")

print(
    "Meses fora de 1–12:",
    df.filter(
        (F.col("released_month") < 1) |
        (F.col("released_month") > 12)
    ).count()
)

print(
    "Dias fora de 1–31:",
    df.filter(
        (F.col("released_day") < 1) |
        (F.col("released_day") > 31)
    ).count()
)

print("\n=== VALORES NUMÉRICOS NEGATIVOS ===")

for campo in [
    "artist_count",
    "in_spotify_playlists",
    "in_spotify_charts",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_charts",
    "bpm"
]:
    negativos = df.filter(F.col(f"`{campo}`") < 0).count()
    print(f"{campo}: {negativos}")

=== PERCENTUAIS FORA DO INTERVALO 0–100 ===
danceability_%: 0
valence_%: 0
energy_%: 0
acousticness_%: 0
instrumentalness_%: 0
liveness_%: 0
speechiness_%: 0

=== DATAS POTENCIALMENTE INVÁLIDAS ===
Meses fora de 1–12: 0
Dias fora de 1–31: 0

=== VALORES NUMÉRICOS NEGATIVOS ===
artist_count: 0
in_spotify_playlists: 0
in_spotify_charts: 0
in_apple_playlists: 0
in_apple_charts: 0
in_deezer_charts: 0
bpm: 0


In [0]:
# ============================================================
# QUALIDADE — CAMPOS NUMÉRICOS RECEBIDOS COMO STRING
# ============================================================

campos_string_numericos = [
    "streams",
    "in_deezer_playlists",
    "in_shazam_charts"
]

for campo in campos_string_numericos:
    print(f"\n=== {campo} ===")

    df.select(campo).show(20, truncate=False)

    # Teste de conversão para número após remoção de vírgulas
    teste = df.withColumn(
        "_valor_numerico",
        F.regexp_replace(F.col(campo), ",", "").cast("double")
    )

    nao_convertiveis = teste.filter(
        F.col(campo).isNotNull() &
        F.col("_valor_numerico").isNull()
    ).count()

    print("Valores não nulos que não podem ser convertidos:", nao_convertiveis)


=== streams ===
+----------+
|streams   |
+----------+
|141381703 |
|133716286 |
|140003974 |
|800840817 |
|303236322 |
|183706234 |
|725980112 |
|58149378  |
|95217315  |
|553634067 |
|505671438 |
|58255150  |
|1316855716|
|387570742 |
|2513188493|
|1163093654|
|496795686 |
|30546883  |
|335222234 |
|363369738 |
+----------+
only showing top 20 rows
Valores não nulos que não podem ser convertidos: 0

=== in_deezer_playlists ===
+-------------------+
|in_deezer_playlists|
+-------------------+
|45                 |
|58                 |
|91                 |
|125                |
|87                 |
|88                 |
|43                 |
|30                 |
|48                 |
|66                 |
|54                 |
|21                 |
|745                |
|182                |
|863                |
|161                |
|78                 |
|95                 |
|54                 |
|23                 |
+-------------------+
only showing top 20 rows
Valores não n

In [0]:
# ============================================================
# RESUMO FINAL — CONVERSIBILIDADE DOS CAMPOS STRING
# ============================================================

campos_string_numericos = [
    "streams",
    "in_deezer_playlists",
    "in_shazam_charts"
]

for campo in campos_string_numericos:

    teste = df.withColumn(
        "_valor_numerico",
        F.regexp_replace(F.col(campo), ",", "").cast("double")
    )

    nao_convertiveis = teste.filter(
        F.col(campo).isNotNull() &
        F.col("_valor_numerico").isNull()
    ).count()

    print(f"{campo}: {nao_convertiveis} valor(es) não convertível(is)")

streams: 0 valor(es) não convertível(is)
in_deezer_playlists: 0 valor(es) não convertível(is)
in_shazam_charts: 0 valor(es) não convertível(is)



# Camada Silver — Preparação e Qualidade dos Dados

Antes da construção da camada Silver, foi realizada uma análise de qualidade sobre os **953 registros** armazenados na Bronze. O objetivo desta etapa foi identificar os problemas presentes nos dados antes de definir qualquer tratamento.

Não foram encontrados registros duplicados. Também não foram identificados valores fora do intervalo de 0 a 100 nos atributos percentuais analisados, valores negativos nos principais campos quantitativos ou meses e dias fora dos intervalos básicos esperados.

Foram encontrados dois atributos com valores ausentes:

- `key`: **95 registros (9,97%)**;
- `in_shazam_charts`: **50 registros (5,25%)**.

Também foi observado que `streams`, `in_deezer_playlists` e `in_shazam_charts` foram recebidos como texto. Parte desses valores utiliza vírgula como separador de milhar, o que exige tratamento antes da conversão para tipos numéricos.

## Análise do atributo `streams`

Durante a validação de `streams`, foi identificado **1 registro com conteúdo incompatível com o valor numérico esperado**. O problema ocorre na música **Love Grows (Where My Rosemary Goes)**, de **Edison Lighthouse**.

Nesse registro, o campo destinado ao número de streams contém informações de outros atributos musicais concatenadas, indicando um problema existente no dado de origem.

Como não há informação suficiente na base para reconstruir de forma confiável o número correto de streams, a música será preservada na camada Silver e apenas o valor inválido de `streams` será convertido para `NULL`.

Essa decisão evita tanto a exclusão desnecessária do registro quanto a criação de um valor que não possa ser sustentado pelos dados disponíveis.

## Análise dos valores ausentes em `in_shazam_charts`

Os **50 registros** sem informação em `in_shazam_charts` também foram analisados antes da definição do tratamento.

A análise mostrou que esse grupo contém músicas com atividade relevante nas demais plataformas, incluindo faixas como **As It Was**, **Blinding Lights**, **Heat Waves**, **Someone You Loved** e **Watermelon Sugar**.

As músicas com informação de Shazam apresentaram média aproximada de **469 milhões de streams**, enquanto o grupo com informação ausente apresentou média aproximada de **1,324 bilhão de streams**.

Esses resultados mostram que a ausência de `in_shazam_charts` não pode ser interpretada como ausência de desempenho ou como valor igual a zero. Por esse motivo, os valores serão preservados como `NULL`, representando informação não disponível na fonte.

## Tratamentos definidos para a camada Silver

Com base no diagnóstico realizado na Bronze, foram definidas as seguintes transformações:

1. padronização dos nomes das colunas;
2. remoção dos separadores de milhar dos campos numéricos recebidos como texto;
3. conversão de `streams`, `in_deezer_playlists` e `in_shazam_charts` para tipos numéricos;
4. conversão para `NULL` do único valor de `streams` incompatível com o domínio numérico, preservando o restante do registro;
5. manutenção dos **95 valores ausentes de `key`** como `NULL`, pois a ausência da tonalidade não impede as demais análises;
6. manutenção dos **50 valores ausentes de `in_shazam_charts`** como `NULL`, sem substituição por zero;
7. criação de uma única data de lançamento a partir dos atributos de ano, mês e dia;
8. manutenção das regras de domínio verificadas para percentuais, datas e valores quantitativos;
9. preservação dos metadados necessários à rastreabilidade entre as camadas Bronze e Silver.

Dessa forma, a camada Silver manterá os **953 registros**, tratando os problemas identificados sem eliminar observações que ainda contenham informações úteis para as análises posteriores.

In [0]:
# ============================================================
# IDENTIFICAÇÃO DE REGISTROS INVÁLIDOS EM STREAMS
# ============================================================

from pyspark.sql import functions as F

# Conversão tolerante: valores inválidos tornam-se NULL
df_teste_streams = df.withColumn(
    "streams_num",
    F.expr(
        "try_cast(regexp_replace(streams, ',', '') AS BIGINT)"
    )
)

# Seleciona somente registros preenchidos que não puderam ser convertidos
df_streams_invalidos = df_teste_streams.filter(
    F.col("streams").isNotNull() &
    F.col("streams_num").isNull()
)

print(
    "Quantidade de registros inválidos em streams:",
    df_streams_invalidos.count()
)

display(
    df_streams_invalidos.select(
        "track_name",
        "artist(s)_name",
        "streams",
        "released_year",
        "released_month",
        "released_day"
    )
)

Quantidade de registros inválidos em streams: 1


track_name,artist(s)_name,streams,released_year,released_month,released_day
Love Grows (Where My Rosemary Goes),Edison Lighthouse,BPM110KeyAModeMajorDanceability53Valence75Energy69Acousticness7Instrumentalness0Liveness17Speechiness3,1970,1,1


In [0]:
# ============================================================
# ANÁLISE DOS VALORES AUSENTES — SHAZAM CHARTS
# ============================================================

# Conversão segura de streams
df_analise = df.withColumn(
    "streams_num",
    F.expr(
        "try_cast(regexp_replace(streams, ',', '') AS BIGINT)"
    )
)

# Criação dos grupos
df_analise = df_analise.withColumn(
    "grupo_shazam",
    F.when(
        F.col("in_shazam_charts").isNull(),
        "Shazam ausente"
    ).otherwise("Shazam informado")
)

# Comparação entre os grupos
resultado_shazam = (
    df_analise
    .groupBy("grupo_shazam")
    .agg(
        F.count("*").alias("quantidade"),
        F.round(F.avg("streams_num"), 0).alias("media_streams"),
        F.min("streams_num").alias("min_streams"),
        F.max("streams_num").alias("max_streams")
    )
)

display(resultado_shazam)

# Amostra dos registros sem informação de Shazam
display(
    df_analise
    .filter(F.col("in_shazam_charts").isNull())
    .select(
        "track_name",
        "artist(s)_name",
        "streams",
        "in_spotify_charts",
        "in_apple_charts",
        "in_shazam_charts"
    )
    .limit(20)
)

grupo_shazam,quantidade,media_streams,min_streams,max_streams
Shazam informado,903,4.69232351E8,2762,3562543890
Shazam ausente,50,1.324224964E9,140430339,3703895074


track_name,artist(s)_name,streams,in_spotify_charts,in_apple_charts,in_shazam_charts
As It Was,Harry Styles,2513188493,130,198,null
Another Love,Tom Odell,1813673666,83,122,null
Blinding Lights,The Weeknd,3703895074,69,199,null
Heat Waves,Glass Animals,2557975762,63,144,null
Sweater Weather,The Neighbourhood,2282771485,61,87,null
Someone You Loved,Lewis Capaldi,2887241814,53,125,null
Watermelon Sugar,Harry Styles,2322580122,34,115,null
Ghost,Justin Bieber,1167330737,24,38,null
Under The Influence,Chris Brown,929964809,26,181,null
Night Changes,One Direction,1131090940,18,20,null


In [0]:
# ============================================================
# CAMADA SILVER — LIMPEZA, TIPAGEM E PADRONIZAÇÃO
# ============================================================

from pyspark.sql import functions as F

# Leitura da Bronze persistida
df_silver = spark.table("spotify_bronze")

# 1. Padronização do nome da coluna com caracteres especiais
df_silver = df_silver.withColumnRenamed(
    "artist(s)_name",
    "artist_name"
)

# 2. Conversão segura dos campos numéricos recebidos como texto
# Valores inválidos são convertidos para NULL
for campo in [
    "streams",
    "in_deezer_playlists",
    "in_shazam_charts"
]:
    df_silver = df_silver.withColumn(
        campo,
        F.expr(
            f"try_cast(regexp_replace(`{campo}`, ',', '') AS BIGINT)"
        )
    )

# 3. Criação da data completa de lançamento
df_silver = df_silver.withColumn(
    "release_date",
    F.make_date(
        F.col("released_year"),
        F.col("released_month"),
        F.col("released_day")
    )
)

# 4. Metadado da transformação para Silver
df_silver = df_silver.withColumn(
    "_data_transformacao_silver",
    F.current_timestamp()
)

# Validação inicial
print("Registros na Silver:", df_silver.count())
print("Colunas na Silver:", len(df_silver.columns))

print(
    "Streams nulos após tratamento:",
    df_silver.filter(F.col("streams").isNull()).count()
)

print(
    "Shazam nulos preservados:",
    df_silver.filter(F.col("in_shazam_charts").isNull()).count()
)

print(
    "Key nulos preservados:",
    df_silver.filter(F.col("key").isNull()).count()
)

print(
    "Datas de lançamento inválidas/nulas:",
    df_silver.filter(F.col("release_date").isNull()).count()
)

display(
    df_silver.select(
        "track_name",
        "artist_name",
        "release_date",
        "streams",
        "in_deezer_playlists",
        "in_shazam_charts",
        "key"
    ).limit(20)
)

Registros na Silver: 953
Colunas na Silver: 29
Streams nulos após tratamento: 1
Shazam nulos preservados: 50
Key nulos preservados: 95
Datas de lançamento inválidas/nulas: 0


track_name,artist_name,release_date,streams,in_deezer_playlists,in_shazam_charts,key
Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2023-07-14,141381703,45,826,B
LALA,Myke Towers,2023-03-23,133716286,58,382,C#
vampire,Olivia Rodrigo,2023-06-30,140003974,91,949,F
Cruel Summer,Taylor Swift,2019-08-23,800840817,125,548,A
WHERE SHE GOES,Bad Bunny,2023-05-18,303236322,87,425,A
Sprinter,"Dave, Central Cee",2023-06-01,183706234,88,946,C#
Ella Baila Sola,"Eslabon Armado, Peso Pluma",2023-03-16,725980112,43,418,F
Columbia,Quevedo,2023-07-07,58149378,30,194,F
fukumean,Gunna,2023-05-15,95217315,48,953,C#
La Bebe - Remix,"Peso Pluma, Yng Lvcas",2023-03-17,553634067,66,339,D


In [0]:
# ============================================================
# PERSISTÊNCIA DA CAMADA SILVER
# ============================================================

tabela_silver = "spotify_silver"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(tabela_silver)
)

# Leitura da tabela persistida para validação
df_silver_salva = spark.table(tabela_silver)

print("Tabela criada:", tabela_silver)
print("Registros persistidos:", df_silver_salva.count())
print("Colunas persistidas:", len(df_silver_salva.columns))

print(
    "Streams nulos:",
    df_silver_salva.filter(F.col("streams").isNull()).count()
)

print(
    "Shazam nulos:",
    df_silver_salva.filter(F.col("in_shazam_charts").isNull()).count()
)

print(
    "Key nulos:",
    df_silver_salva.filter(F.col("key").isNull()).count()
)

display(df_silver_salva.limit(10))

Tabela criada: spotify_silver
Registros persistidos: 953
Colunas persistidas: 29
Streams nulos: 1
Shazam nulos: 50
Key nulos: 95


track_name,artist_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,in_apple_charts,in_deezer_playlists,in_deezer_charts,in_shazam_charts,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%,_data_ingestao,_arquivo_origem,_url_origem,release_date,_data_transformacao_silver
Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,263,45,10,826,125,B,Major,80,89,83,31,0,8,4,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv,2023-07-14,2026-09-27T18:31:54.487Z
LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,126,58,14,382,92,C#,Major,71,61,74,7,0,10,4,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv,2023-03-23,2026-09-27T18:31:54.487Z
vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,207,91,14,949,138,F,Major,51,32,53,17,0,31,6,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv,2023-06-30,2026-09-27T18:31:54.487Z
Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,207,125,12,548,170,A,Major,55,58,72,11,0,11,15,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv,2019-08-23,2026-09-27T18:31:54.487Z
WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,133,87,15,425,144,A,Minor,65,23,80,14,63,11,6,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv,2023-05-18,2026-09-27T18:31:54.487Z
Sprinter,"Dave, Central Cee",2,2023,6,1,2186,91,183706234,67,213,88,17,946,141,C#,Major,92,66,58,19,0,8,24,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv,2023-06-01,2026-09-27T18:31:54.487Z
Ella Baila Sola,"Eslabon Armado, Peso Pluma",2,2023,3,16,3090,50,725980112,34,222,43,13,418,148,F,Minor,67,83,76,48,0,8,3,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv,2023-03-16,2026-09-27T18:31:54.487Z
Columbia,Quevedo,1,2023,7,7,714,43,58149378,25,89,30,13,194,100,F,Major,67,26,71,37,0,11,4,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv,2023-07-07,2026-09-27T18:31:54.487Z
fukumean,Gunna,1,2023,5,15,1096,83,95217315,60,210,48,11,953,130,C#,Minor,85,22,62,12,0,28,9,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv,2023-05-15,2026-09-27T18:31:54.487Z
La Bebe - Remix,"Peso Pluma, Yng Lvcas",2,2023,3,17,2953,44,553634067,49,110,66,13,339,170,D,Minor,81,56,48,21,0,8,33,2026-09-27T18:31:10.525Z,spotify-2023.csv,https://raw.githubusercontent.com/lizbrandao/PUCRJ/main/spotify-2023.csv,2023-03-17,2026-09-27T18:31:54.487Z


In [0]:
# ============================================================
# VALIDAÇÃO FINAL — BRONZE x SILVER
# ============================================================

from pyspark.sql import functions as F

bronze = spark.table("spotify_bronze")
silver = spark.table("spotify_silver")

# 1. Controle de registros
print("=== CONTROLE DE REGISTROS ===")
print("Bronze:", bronze.count())
print("Silver:", silver.count())
print("Diferença:", bronze.count() - silver.count())

# 2. Alterações de estrutura
print("\n=== ALTERAÇÕES DE ESTRUTURA ===")
print("Colunas Bronze:", len(bronze.columns))
print("Colunas Silver:", len(silver.columns))

# 3. Tipos após tratamento
print("\n=== TIPOS APÓS TRATAMENTO ===")

for campo in [
    "streams",
    "in_deezer_playlists",
    "in_shazam_charts",
    "release_date"
]:
    print(f"{campo}: {dict(silver.dtypes).get(campo)}")

# 4. Valores ausentes preservados ou tratados
print("\n=== VALORES AUSENTES PRESERVADOS/TRATADOS ===")
print(
    "streams NULL:",
    silver.filter(F.col("streams").isNull()).count()
)
print(
    "key NULL:",
    silver.filter(F.col("key").isNull()).count()
)
print(
    "in_shazam_charts NULL:",
    silver.filter(F.col("in_shazam_charts").isNull()).count()
)

# 5. Validação de duplicados
print("\n=== DUPLICADOS NA SILVER ===")

colunas_negocio = [
    c for c in silver.columns
    if not c.startswith("_")
]

total_silver = silver.count()

distintos_silver = (
    silver
    .select(*colunas_negocio)
    .distinct()
    .count()
)

print("Total de registros:", total_silver)
print("Registros distintos:", distintos_silver)
print("Duplicados:", total_silver - distintos_silver)

=== CONTROLE DE REGISTROS ===
Bronze: 953
Silver: 953
Diferença: 0

=== ALTERAÇÕES DE ESTRUTURA ===
Colunas Bronze: 27
Colunas Silver: 29

=== TIPOS APÓS TRATAMENTO ===
streams: bigint
in_deezer_playlists: bigint
in_shazam_charts: bigint
release_date: date

=== VALORES AUSENTES PRESERVADOS/TRATADOS ===
streams NULL: 1
key NULL: 95
in_shazam_charts NULL: 50

=== DUPLICADOS NA SILVER ===
Total de registros: 953
Registros distintos: 953
Duplicados: 0



# Camada Gold — Modelagem Analítica

Com os dados tratados e validados na camada Silver, a etapa seguinte consiste em organizá-los para responder às perguntas definidas no início do projeto.

A camada Gold será construída utilizando uma modelagem dimensional. O objetivo é separar as informações descritivas das métricas de desempenho, facilitando as consultas e evitando ambiguidades nas análises.

## Grão do modelo

O desempenho será analisado principalmente no nível da **música**. Cada registro da tabela fato representará uma música presente no conjunto de dados tratado.

Entretanto, uma mesma música pode possuir mais de um artista. Como essa relação é muitos-para-muitos, os artistas não serão simplesmente duplicados na tabela fato.

A análise da base identificou **366 músicas com mais de um artista**. A separação dos nomes utilizando a vírgula como delimitador foi consistente com `artist_count` em 952 dos 953 registros. Foi encontrada uma única inconsistência, na qual `artist_count` indica sete artistas, enquanto apenas seis puderam ser identificados no conteúdo disponível. Nesse caso, serão considerados somente os artistas efetivamente identificáveis, sem reconstrução de informação não disponível na fonte.

## Identificação das músicas

O dataset não possui um identificador único explícito para cada música. Por esse motivo, antes da construção das dimensões foi verificada a possibilidade de utilizar os atributos disponíveis para identificar cada registro.

O uso isolado de `track_name` não foi considerado adequado, pois foram encontrados **943 nomes distintos para 953 registros**, com 10 nomes de músicas repetidos.

Também foi testada a combinação entre nome da música, artista e data de lançamento. Essa alternativa resultou em **950 combinações distintas**, permanecendo três combinações repetidas: `SNAP`, de Rosa Linn; `SPIT IN MY FACE!`, de ThxSoMch; e `Take My Breath`, de The Weeknd.

A comparação desses registros mostrou diferenças não apenas nas métricas de desempenho, mas também em algumas características musicais. Por isso, as métricas de streams, playlists e charts não serão utilizadas para definir a identidade da música.

Foi então avaliada uma combinação formada pelos atributos descritivos e musicais: nome da faixa, artista, data de lançamento, BPM, tonalidade, modo, danceability, valence, energy, acousticness, instrumentalness, liveness e speechiness.

Essa combinação apresentou **953 identidades distintas para os 953 registros**, sem repetições.

Com base nessa validação, será criada uma chave técnica `musica_id` a partir desses atributos. Dessa forma, cada registro poderá ser identificado de maneira estável dentro deste conjunto de dados sem incorporar métricas de desempenho à sua identidade.

## Estrutura proposta

### `dim_musica`

Representará as músicas e seus atributos descritivos e musicais, como:

- nome da faixa;
- BPM;
- tonalidade e modo;
- danceability;
- valence;
- energy;
- acousticness;
- instrumentalness;
- liveness;
- speechiness.

### `dim_artista`

Conterá uma ocorrência única para cada artista identificado na base.

### `bridge_musica_artista`

Representará a relação entre músicas e artistas.

Essa tabela é necessária porque o dataset registra colaborações em um mesmo campo e **366 músicas possuem mais de um artista**. A estrutura permitirá analisar artistas individualmente sem alterar o grão da tabela fato.

### `dim_tempo`

Organizará as informações relacionadas à data de lançamento das músicas, permitindo análises por data, ano, mês e outros recortes temporais.

### `fato_desempenho_musica`

Concentrará as métricas utilizadas para analisar o desempenho das músicas, incluindo:

- streams;
- presença em playlists;
- presença em charts das plataformas disponíveis na base.

## Relacionamentos

A estrutura analítica principal será:

**dim_musica → fato_desempenho_musica ← dim_tempo**

A relação entre músicas e artistas será representada separadamente:

**dim_musica → bridge_musica_artista ← dim_artista**

Essa estrutura permite preservar o grão da música e, ao mesmo tempo, realizar análises individuais por artista.

## Objetivo da camada Gold

A modelagem será utilizada para responder às cinco primeiras perguntas do projeto relacionadas a desempenho, exposição, período de lançamento, características musicais e concentração do consumo.

A sexta pergunta, relacionada à recepção crítica e à avaliação dos usuários, dependerá da etapa posterior de enriquecimento com uma fonte externa.

In [0]:
# ============================================================
# GOLD — ANÁLISE DA RELAÇÃO MÚSICA x ARTISTA
# ============================================================

from pyspark.sql import functions as F

silver = spark.table("spotify_silver")

# Separa os artistas usando a vírgula como delimitador
df_artistas_analise = (
    silver
    .withColumn(
        "artistas_separados",
        F.split(F.col("artist_name"), ",")
    )
    .withColumn(
        "qtd_artistas_separados",
        F.size(F.col("artistas_separados"))
    )
)

# 1. Distribuição da quantidade de artistas
print("=== DISTRIBUIÇÃO DA QUANTIDADE DE ARTISTAS ===")

display(
    df_artistas_analise
    .groupBy("artist_count", "qtd_artistas_separados")
    .count()
    .orderBy("artist_count", "qtd_artistas_separados")
)

# 2. Consistência entre artist_count e separação por vírgula
print("=== CONSISTÊNCIA DO ARTIST_COUNT ===")

inconsistencias = df_artistas_analise.filter(
    F.col("artist_count") != F.col("qtd_artistas_separados")
)

print(
    "Registros em que artist_count difere da separação por vírgula:",
    inconsistencias.count()
)

display(
    inconsistencias.select(
        "track_name",
        "artist_name",
        "artist_count",
        "qtd_artistas_separados"
    ).limit(20)
)

# 3. Músicas com múltiplos artistas
print("=== MÚSICAS COM MAIS DE UM ARTISTA ===")

multiplos_artistas = df_artistas_analise.filter(
    F.col("artist_count") > 1
)

print(
    "Quantidade de músicas com múltiplos artistas:",
    multiplos_artistas.count()
)

display(
    multiplos_artistas.select(
        "track_name",
        "artist_name",
        "artist_count"
    ).limit(20)
)

=== DISTRIBUIÇÃO DA QUANTIDADE DE ARTISTAS ===


artist_count,qtd_artistas_separados,count
1,1,587
2,2,254
3,3,85
4,4,15
5,5,5
6,6,3
7,6,1
7,7,1
8,8,2


=== CONSISTÊNCIA DO ARTIST_COUNT ===
Registros em que artist_count difere da separação por vírgula: 1


track_name,artist_name,artist_count,qtd_artistas_separados
"Cayï¿½ï¿½ La Noche (feat. Cruz Cafunï¿½ï¿½, Abhir Hathi, Bejo, EL IMA)","Quevedo, La Pantera, Juseph, Cruz Cafunï¿½ï¿½, Bï¿½ï¿½jo, Abhir Hathi",7,6


=== MÚSICAS COM MAIS DE UM ARTISTA ===
Quantidade de músicas com múltiplos artistas: 366


track_name,artist_name,artist_count
Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2
Sprinter,"Dave, Central Cee",2
Ella Baila Sola,"Eslabon Armado, Peso Pluma",2
La Bebe - Remix,"Peso Pluma, Yng Lvcas",2
un x100to,"Bad Bunny, Grupo Frontera",2
Classy 101,"Feid, Young Miko",2
LADY GAGA,"Gabito Ballesteros, Junior H, Peso Pluma",3
"Peso Pluma: Bzrp Music Sessions, Vol. 55","Bizarrap, Peso Pluma",2
Popular (with Playboi Carti & Madonna) - The Idol Vol. 1 (Music from the HBO Original Series),"The Weeknd, Madonna, Playboi Carti",3
Calm Down (with Selena Gomez),"Rï¿½ï¿½ma, Selena G",2


In [0]:
# GOLD — ANÁLISE DA RELAÇÃO MÚSICA x ARTISTA
# ============================================================

from pyspark.sql import functions as F

silver = spark.table("spotify_silver")

# Separação dos artistas utilizando a vírgula como delimitador
df_artistas_analise = (
    silver
    .withColumn(
        "artistas_separados",
        F.split(F.col("artist_name"), ",")
    )
    .withColumn(
        "qtd_artistas_separados",
        F.size(F.col("artistas_separados"))
    )
)

# Distribuição da quantidade de artistas
print("=== DISTRIBUIÇÃO DA QUANTIDADE DE ARTISTAS ===")

display(
    df_artistas_analise
    .groupBy("artist_count", "qtd_artistas_separados")
    .count()
    .orderBy("artist_count", "qtd_artistas_separados")
)

# Validação da consistência da separação
inconsistencias = df_artistas_analise.filter(
    F.col("artist_count") != F.col("qtd_artistas_separados")
)

print("=== CONSISTÊNCIA DO ARTIST_COUNT ===")
print(
    "Registros em que artist_count difere da separação por vírgula:",
    inconsistencias.count()
)

# Detalhamento das inconsistências encontradas
print("=== DETALHAMENTO DAS INCONSISTÊNCIAS ===")

display(
    inconsistencias.select(
        "track_name",
        "artist_name",
        "artist_count",
        "qtd_artistas_separados",
        "artistas_separados"
    )
)

# Músicas com múltiplos artistas
multiplos_artistas = df_artistas_analise.filter(
    F.col("artist_count") > 1
)

print("=== MÚSICAS COM MAIS DE UM ARTISTA ===")
print(
    "Quantidade de músicas com múltiplos artistas:",
    multiplos_artistas.count()
)

display(
    multiplos_artistas.select(
        "track_name",
        "artist_name",
        "artist_count"
    ).limit(20)
)

=== DISTRIBUIÇÃO DA QUANTIDADE DE ARTISTAS ===


artist_count,qtd_artistas_separados,count
1,1,587
2,2,254
3,3,85
4,4,15
5,5,5
6,6,3
7,6,1
7,7,1
8,8,2


=== CONSISTÊNCIA DO ARTIST_COUNT ===
Registros em que artist_count difere da separação por vírgula: 1
=== DETALHAMENTO DAS INCONSISTÊNCIAS ===


track_name,artist_name,artist_count,qtd_artistas_separados,artistas_separados
"Cayï¿½ï¿½ La Noche (feat. Cruz Cafunï¿½ï¿½, Abhir Hathi, Bejo, EL IMA)","Quevedo, La Pantera, Juseph, Cruz Cafunï¿½ï¿½, Bï¿½ï¿½jo, Abhir Hathi",7,6,"List(Quevedo, La Pantera, Juseph, Cruz Cafunï¿½ï¿½, Bï¿½ï¿½jo, Abhir Hathi)"


=== MÚSICAS COM MAIS DE UM ARTISTA ===
Quantidade de músicas com múltiplos artistas: 366


track_name,artist_name,artist_count
Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2
Sprinter,"Dave, Central Cee",2
Ella Baila Sola,"Eslabon Armado, Peso Pluma",2
La Bebe - Remix,"Peso Pluma, Yng Lvcas",2
un x100to,"Bad Bunny, Grupo Frontera",2
Classy 101,"Feid, Young Miko",2
LADY GAGA,"Gabito Ballesteros, Junior H, Peso Pluma",3
"Peso Pluma: Bzrp Music Sessions, Vol. 55","Bizarrap, Peso Pluma",2
Popular (with Playboi Carti & Madonna) - The Idol Vol. 1 (Music from the HBO Original Series),"The Weeknd, Madonna, Playboi Carti",3
Calm Down (with Selena Gomez),"Rï¿½ï¿½ma, Selena G",2


In [0]:
# ============================================================
# GOLD — VALIDAÇÃO DA CHAVE DA MÚSICA
# ============================================================

silver = spark.table("spotify_silver")

total_registros = silver.count()
nomes_distintos = silver.select("track_name").distinct().count()

print("=== VALIDAÇÃO DE UNICIDADE DE TRACK_NAME ===")
print("Total de registros:", total_registros)
print("Nomes de músicas distintos:", nomes_distintos)
print("Diferença:", total_registros - nomes_distintos)

# Identifica nomes de músicas que aparecem mais de uma vez
musicas_repetidas = (
    silver
    .groupBy("track_name")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"), "track_name")
)

print(
    "Quantidade de nomes de músicas repetidos:",
    musicas_repetidas.count()
)

display(musicas_repetidas)

# Mostra os registros completos das músicas com nomes repetidos
display(
    silver
    .join(
        musicas_repetidas.select("track_name"),
        on="track_name",
        how="inner"
    )
    .select(
        "track_name",
        "artist_name",
        "release_date",
        "streams"
    )
    .orderBy("track_name", "artist_name")
)

=== VALIDAÇÃO DE UNICIDADE DE TRACK_NAME ===
Total de registros: 953
Nomes de músicas distintos: 943
Diferença: 10
Quantidade de nomes de músicas repetidos: 10


track_name,count
About Damn Time,2
Daylight,2
Die For You,2
Flowers,2
Let It Snow! Let It Snow! Let It Snow!,2
Miss You,2
Numb,2
SNAP,2
SPIT IN MY FACE!,2
Take My Breath,2


track_name,artist_name,release_date,streams
About Damn Time,Lizzo,2022-04-14,723894473
About Damn Time,Lizzo,2022-07-15,723894473
Daylight,David Kushner,2023-04-14,387570742
Daylight,Harry Styles,2022-05-20,290833204
Die For You,Joji,2022-11-04,246390068
Die For You,The Weeknd,2016-11-24,1647990401
Flowers,Lauren Spencer Smith,2021-08-09,184826429
Flowers,Miley Cyrus,2023-01-12,1316855716
Let It Snow! Let It Snow! Let It Snow!,Dean Martin,1959-11-16,446390129
Let It Snow! Let It Snow! Let It Snow!,"Frank Sinatra, B. Swanson Quartet",1950-01-01,473248298


In [0]:
# ============================================================
# GOLD — VALIDAÇÃO DA CHAVE COMPOSTA DA MÚSICA
# ============================================================

silver = spark.table("spotify_silver")

chave_candidata = [
    "track_name",
    "artist_name",
    "release_date"
]

total_registros = silver.count()

chaves_distintas = (
    silver
    .select(*chave_candidata)
    .distinct()
    .count()
)

print("=== VALIDAÇÃO DA CHAVE CANDIDATA ===")
print("Chave: track_name + artist_name + release_date")
print("Total de registros:", total_registros)
print("Chaves distintas:", chaves_distintas)
print("Diferença:", total_registros - chaves_distintas)

# Procura combinações repetidas
chaves_repetidas = (
    silver
    .groupBy(*chave_candidata)
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Combinações repetidas:",
    chaves_repetidas.count()
)

display(chaves_repetidas)

=== VALIDAÇÃO DA CHAVE CANDIDATA ===
Chave: track_name + artist_name + release_date
Total de registros: 953
Chaves distintas: 950
Diferença: 3
Combinações repetidas: 3


track_name,artist_name,release_date,count
SNAP,Rosa Linn,2022-03-19,2
SPIT IN MY FACE!,ThxSoMch,2022-10-31,2
Take My Breath,The Weeknd,2021-08-06,2


In [0]:
# ============================================================
# GOLD — ANÁLISE DAS CHAVES CANDIDATAS REPETIDAS
# ============================================================

registros_repetidos = (
    silver
    .join(
        chaves_repetidas.select(
            "track_name",
            "artist_name",
            "release_date"
        ),
        on=[
            "track_name",
            "artist_name",
            "release_date"
        ],
        how="inner"
    )
    .orderBy(
        "track_name",
        "artist_name",
        "release_date"
    )
)

print("=== REGISTROS DAS 3 CHAVES REPETIDAS ===")
print("Quantidade de registros envolvidos:", registros_repetidos.count())

display(
    registros_repetidos.select(
        "track_name",
        "artist_name",
        "release_date",
        "streams",
        "in_spotify_playlists",
        "in_spotify_charts",
        "in_apple_playlists",
        "in_apple_charts",
        "in_deezer_playlists",
        "in_deezer_charts",
        "in_shazam_charts"
    )
)

=== REGISTROS DAS 3 CHAVES REPETIDAS ===
Quantidade de registros envolvidos: 6


track_name,artist_name,release_date,streams,in_spotify_playlists,in_spotify_charts,in_apple_playlists,in_apple_charts,in_deezer_playlists,in_deezer_charts,in_shazam_charts
SNAP,Rosa Linn,2022-03-19,711366595,1818,0,3,0,63,0,353
SNAP,Rosa Linn,2022-03-19,726307468,3202,18,148,80,226,24,0
SPIT IN MY FACE!,ThxSoMch,2022-10-31,301869854,573,0,1,0,18,0,24
SPIT IN MY FACE!,ThxSoMch,2022-10-31,303216294,629,14,32,3,9,0,0
Take My Breath,The Weeknd,2021-08-06,130655803,2597,0,17,80,38,0,0
Take My Breath,The Weeknd,2021-08-06,432702334,6392,0,174,73,344,0,0


In [0]:
# ============================================================
# GOLD — COMPARAÇÃO COMPLETA DOS REGISTROS REPETIDOS
# ============================================================

display(
    registros_repetidos
    .drop(
        "_data_ingestao",
        "_arquivo_origem",
        "_url_origem",
        "_data_transformacao_silver"
    )
    .orderBy(
        "track_name",
        "artist_name",
        "release_date"
    )
)

track_name,artist_name,release_date,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,in_apple_charts,in_deezer_playlists,in_deezer_charts,in_shazam_charts,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
SNAP,Rosa Linn,2022-03-19,1,2022,3,19,1818,0,711366595,3,0,63,0,353,170,null,Major,56,52,64,11,0,45,7
SNAP,Rosa Linn,2022-03-19,1,2022,3,19,3202,18,726307468,148,80,226,24,0,170,null,Major,56,53,64,11,0,45,6
SPIT IN MY FACE!,ThxSoMch,2022-10-31,1,2022,10,31,573,0,301869854,1,0,18,0,24,166,C#,Major,70,57,57,9,20,11,7
SPIT IN MY FACE!,ThxSoMch,2022-10-31,1,2022,10,31,629,14,303216294,32,3,9,0,0,94,G#,Major,73,65,79,5,2,11,6
Take My Breath,The Weeknd,2021-08-06,1,2021,8,6,2597,0,130655803,17,80,38,0,0,121,A#,Minor,70,35,77,1,0,26,4
Take My Breath,The Weeknd,2021-08-06,1,2021,8,6,6392,0,432702334,174,73,344,0,0,121,G#,Major,75,53,74,2,0,11,5


In [0]:
# ============================================================
# GOLD — ATRIBUTOS QUE DIFERENCIAM OS REGISTROS REPETIDOS
# ============================================================

from pyspark.sql import functions as F

# Remove metadados técnicos da comparação
colunas_comparacao = [
    c for c in registros_repetidos.columns
    if not c.startswith("_")
]

resultados = []

for musica in ["SNAP", "SPIT IN MY FACE!", "Take My Breath"]:

    dados = registros_repetidos.filter(
        F.col("track_name") == musica
    )

    for coluna in colunas_comparacao:

        valores_distintos = (
            dados
            .select(coluna)
            .distinct()
            .count()
        )

        if valores_distintos > 1:
            resultados.append((musica, coluna))

diferencas = spark.createDataFrame(
    resultados,
    ["musica", "atributo_diferente"]
)

print("=== ATRIBUTOS QUE DIFEREM ENTRE OS REGISTROS ===")

display(
    diferencas.orderBy(
        "musica",
        "atributo_diferente"
    )
)

=== ATRIBUTOS QUE DIFEREM ENTRE OS REGISTROS ===


musica,atributo_diferente
SNAP,in_apple_charts
SNAP,in_apple_playlists
SNAP,in_deezer_charts
SNAP,in_deezer_playlists
SNAP,in_shazam_charts
SNAP,in_spotify_charts
SNAP,in_spotify_playlists
SNAP,speechiness_%
SNAP,streams
SNAP,valence_%


In [0]:
# ============================================================
# GOLD — VALIDAÇÃO DA IDENTIDADE ANALÍTICA DA MÚSICA
# ============================================================

silver = spark.table("spotify_silver")

atributos_identidade = [
    "track_name",
    "artist_name",
    "release_date",
    "bpm",
    "key",
    "mode",
    "danceability_%",
    "valence_%",
    "energy_%",
    "acousticness_%",
    "instrumentalness_%",
    "liveness_%",
    "speechiness_%"
]

total = silver.count()

identidades_distintas = (
    silver
    .select(*atributos_identidade)
    .distinct()
    .count()
)

repetidas = (
    silver
    .groupBy(*atributos_identidade)
    .count()
    .filter(F.col("count") > 1)
)

print("=== IDENTIDADE ANALÍTICA DA MÚSICA ===")
print("Total de registros:", total)
print("Identidades distintas:", identidades_distintas)
print("Diferença:", total - identidades_distintas)
print("Identidades repetidas:", repetidas.count())

display(repetidas)

=== IDENTIDADE ANALÍTICA DA MÚSICA ===
Total de registros: 953
Identidades distintas: 953
Diferença: 0
Identidades repetidas: 0


track_name,artist_name,release_date,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%,count


In [0]:
# ============================================================
# GOLD — DIMENSÃO MÚSICA
# ============================================================

from pyspark.sql import functions as F

silver = spark.table("spotify_silver")

identidade = [
    "track_name", "artist_name", "release_date", "bpm", "key", "mode",
    "danceability_%", "valence_%", "energy_%", "acousticness_%",
    "instrumentalness_%", "liveness_%", "speechiness_%"
]

silver_com_id = silver.withColumn(
    "musica_id",
    F.sha2(
        F.concat_ws(
            "||",
            *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>"))
              for c in identidade]
        ),
        256
    )
)

atributos_musica = [
    "musica_id", "track_name", "bpm", "key", "mode",
    "danceability_%", "valence_%", "energy_%",
    "acousticness_%", "instrumentalness_%",
    "liveness_%", "speechiness_%"
]

dim_musica = (
    silver_com_id
    .select(*atributos_musica)
    .dropDuplicates(["musica_id"])
)

print("=== DIM_MUSICA ===")
print("Registros:", dim_musica.count())
print("IDs distintos:", dim_musica.select("musica_id").distinct().count())
print("IDs nulos:", dim_musica.filter(F.col("musica_id").isNull()).count())

display(dim_musica.limit(10))

=== DIM_MUSICA ===
Registros: 953
IDs distintos: 953
IDs nulos: 0


musica_id,track_name,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
ad2b90151cde3104c5c7177d07cd37423bf212538b1eb6349e16c36a70e8e028,Seven (feat. Latto) (Explicit Ver.),125,B,Major,80,89,83,31,0,8,4
bb3df471ead760e88d9bf04a5e11b58f2b8dc84d8bafeaecd3b9646b735c1937,LALA,92,C#,Major,71,61,74,7,0,10,4
966d518d0b853fdd7f06866b9ba07895b0e54b95c5c9ec3283d2d2e8634bb4b7,vampire,138,F,Major,51,32,53,17,0,31,6
92ad76b822f03f90c1b4529e1c1479d2176d3d757797d50833b5f137a29f8455,Cruel Summer,170,A,Major,55,58,72,11,0,11,15
fbd0644263fd1da92dc2cc37cb321598608f23dc1d05548a1d0ea0345a9b135b,WHERE SHE GOES,144,A,Minor,65,23,80,14,63,11,6
4652eecd2c19965e2357a5e35fcee9e21e9dd40a5412dde81c9bcf1cfe155b1d,Sprinter,141,C#,Major,92,66,58,19,0,8,24
af086a918f5e3ff7149cfb3a116a46d5317dad5a45c395d472fd66a7c9c6b32e,Ella Baila Sola,148,F,Minor,67,83,76,48,0,8,3
3f53b86488768af60e7d5bbb27fe4c553dfe85d8e2e576eb4d1b32c8aef29bb4,Columbia,100,F,Major,67,26,71,37,0,11,4
3a083b1e50e6cd7b7bbaf70957cf91256245db6465db1b201cb020c4ec9a4fb3,fukumean,130,C#,Minor,85,22,62,12,0,28,9
12fb0991d47a426fe038fcf2b8789a625de8f7cd73000939dcc5ecc22fcd3db9,La Bebe - Remix,170,D,Minor,81,56,48,21,0,8,33


In [0]:
# ============================================================
# GOLD — DIMENSÃO ARTISTA
# ============================================================

musica_artista = (
    silver_com_id
    .select(
        "musica_id",
        F.explode(
            F.split(F.col("artist_name"), ",")
        ).alias("artist_name")
    )
    .withColumn(
        "artist_name",
        F.trim("artist_name")
    )
)

dim_artista = (
    musica_artista
    .select("artist_name")
    .distinct()
    .withColumn(
        "artista_id",
        F.sha2(F.col("artist_name"), 256)
    )
    .select("artista_id", "artist_name")
)

print("=== DIM_ARTISTA ===")
print("Artistas:", dim_artista.count())
print("IDs distintos:", dim_artista.select("artista_id").distinct().count())
print("IDs nulos:", dim_artista.filter(F.col("artista_id").isNull()).count())

display(dim_artista.limit(10))

=== DIM_ARTISTA ===
Artistas: 699
IDs distintos: 699
IDs nulos: 0


artista_id,artist_name
6cd1c0e957072fbbf481a455719dd79df6b044644d89ee87c937722e68977caa,Latto
a0b0b79c90af400d012c20ff4e5190d46ea6da7d00a9f43f50766524455a16b4,21 Savage
7ff034b092dab1be2452f806f15aec2c9052f0efeb7042c986abc58af9a21ebd,FMK
a435e2d26d33b5972b6638363c8dedb38f25166b363b273d7a59fe44975e49b0,Tyler
7a4721a879cb49bf793cbf62a831c2e70ae16da6a6a56fc83fe0dcdccaacee97,Aqua
bb647e29b34334079a9d553184f6cb58abd93182f8a4f7e95caf4bea0c2b2171,dennis
c6890ee14c54197368f0f86880d334030858d88bbdda3390119e8c3261387a29,MC Kevin o Chris
942ed3a371fa9c714abb580207ad665a7a9314a6475abcceed46c620f1fa2284,Sam Smith
4de4307a6586354552fe4cbefb05724ddc571aa06988a0deb8534a41465cbeab,Ruth B.
ff21a257e0321eaa4fc4acfaef76c6d12621c8a636462a433c6370abe1895eea,Ellie Goulding


In [0]:
# ============================================================
# GOLD — BRIDGE MÚSICA-ARTISTA
# ============================================================

bridge_musica_artista = (
    musica_artista
    .join(
        dim_artista,
        on="artist_name",
        how="inner"
    )
    .select(
        "musica_id",
        "artista_id"
    )
    .dropDuplicates()
)

print("=== BRIDGE_MUSICA_ARTISTA ===")
print("Relações:", bridge_musica_artista.count())
print(
    "Músicas representadas:",
    bridge_musica_artista.select("musica_id").distinct().count()
)
print(
    "Artistas representados:",
    bridge_musica_artista.select("artista_id").distinct().count()
)

display(bridge_musica_artista.limit(20))

=== BRIDGE_MUSICA_ARTISTA ===
Relações: 1479
Músicas representadas: 953
Artistas representados: 699


musica_id,artista_id
ad2b90151cde3104c5c7177d07cd37423bf212538b1eb6349e16c36a70e8e028,6cd1c0e957072fbbf481a455719dd79df6b044644d89ee87c937722e68977caa
85f3544db35b25e7f16d6eb9ef8e906f9cba3e738945a657b6fde7f6b1107435,a0b0b79c90af400d012c20ff4e5190d46ea6da7d00a9f43f50766524455a16b4
59d6e1497e138e6dc6ce4b9c3fc17b99e766515ad8848c9fce640adcf4f593a6,7ff034b092dab1be2452f806f15aec2c9052f0efeb7042c986abc58af9a21ebd
93da391ea1c337c35b2f9bb28441bb16ea7b172f41939e00ffd9059994b56b06,a435e2d26d33b5972b6638363c8dedb38f25166b363b273d7a59fe44975e49b0
3178b017c8e863a1d3f07b321e0bd79ac5253f5d74b70c191ed15486213a298a,7a4721a879cb49bf793cbf62a831c2e70ae16da6a6a56fc83fe0dcdccaacee97
2ce5981cdd5514be63fcdaf9ca6475859dd8979489f1e17ae99ccbd3db200c18,bb647e29b34334079a9d553184f6cb58abd93182f8a4f7e95caf4bea0c2b2171
2ce5981cdd5514be63fcdaf9ca6475859dd8979489f1e17ae99ccbd3db200c18,c6890ee14c54197368f0f86880d334030858d88bbdda3390119e8c3261387a29
818246fae2dd3383c09923aa354859942aa11cbc64b11f6f2514bc19189f5992,942ed3a371fa9c714abb580207ad665a7a9314a6475abcceed46c620f1fa2284
e5ed9083e8105130ebf2c93ce0f485678cfc0f7b3784c8be09f513777ed3de5d,4de4307a6586354552fe4cbefb05724ddc571aa06988a0deb8534a41465cbeab
4a874d4f823e1cfb20264c50379bd6137a7e026020f9f0a6242b8de5a8f1b481,ff21a257e0321eaa4fc4acfaef76c6d12621c8a636462a433c6370abe1895eea


In [0]:
# ============================================================
# GOLD — DIMENSÃO TEMPO
# ============================================================

dim_tempo = (
    silver_com_id
    .select(
        F.col("release_date").alias("data_id"),
        F.col("released_year").alias("ano"),
        F.col("released_month").alias("mes"),
        F.col("released_day").alias("dia")
    )
    .distinct()
    .orderBy("data_id")
)

print("=== DIM_TEMPO ===")
print("Datas distintas:", dim_tempo.count())
print(
    "Datas nulas:",
    dim_tempo.filter(F.col("data_id").isNull()).count()
)

display(dim_tempo.limit(20))

=== DIM_TEMPO ===
Datas distintas: 477
Datas nulas: 0


data_id,ano,mes,dia
1930-01-01,1930,1,1
1942-01-01,1942,1,1
1946-11-01,1946,11,1
1950-01-01,1950,1,1
1952-01-01,1952,1,1
1957-01-01,1957,1,1
1958-01-01,1958,1,1
1959-01-01,1959,1,1
1959-11-16,1959,11,16
1963-10-14,1963,10,14


In [0]:
# ============================================================
# GOLD — FATO DESEMPENHO DA MÚSICA
# ============================================================

campos_fato = [
    "musica_id", "release_date", "streams",
    "in_spotify_playlists", "in_spotify_charts",
    "in_apple_playlists", "in_apple_charts",
    "in_deezer_playlists", "in_deezer_charts",
    "in_shazam_charts"
]

fato_desempenho_musica = (
    silver_com_id
    .select(*campos_fato)
    .withColumnRenamed("release_date", "data_id")
)

print("=== FATO_DESEMPENHO_MUSICA ===")
print("Registros:", fato_desempenho_musica.count())
print("Músicas:", fato_desempenho_musica.select("musica_id").distinct().count())
print("musica_id nulos:", fato_desempenho_musica.filter(F.col("musica_id").isNull()).count())
print("data_id nulos:", fato_desempenho_musica.filter(F.col("data_id").isNull()).count())

display(fato_desempenho_musica.limit(10))

=== FATO_DESEMPENHO_MUSICA ===
Registros: 953
Músicas: 953
musica_id nulos: 0
data_id nulos: 0


musica_id,data_id,streams,in_spotify_playlists,in_spotify_charts,in_apple_playlists,in_apple_charts,in_deezer_playlists,in_deezer_charts,in_shazam_charts
ad2b90151cde3104c5c7177d07cd37423bf212538b1eb6349e16c36a70e8e028,2023-07-14,141381703,553,147,43,263,45,10,826
bb3df471ead760e88d9bf04a5e11b58f2b8dc84d8bafeaecd3b9646b735c1937,2023-03-23,133716286,1474,48,48,126,58,14,382
966d518d0b853fdd7f06866b9ba07895b0e54b95c5c9ec3283d2d2e8634bb4b7,2023-06-30,140003974,1397,113,94,207,91,14,949
92ad76b822f03f90c1b4529e1c1479d2176d3d757797d50833b5f137a29f8455,2019-08-23,800840817,7858,100,116,207,125,12,548
fbd0644263fd1da92dc2cc37cb321598608f23dc1d05548a1d0ea0345a9b135b,2023-05-18,303236322,3133,50,84,133,87,15,425
4652eecd2c19965e2357a5e35fcee9e21e9dd40a5412dde81c9bcf1cfe155b1d,2023-06-01,183706234,2186,91,67,213,88,17,946
af086a918f5e3ff7149cfb3a116a46d5317dad5a45c395d472fd66a7c9c6b32e,2023-03-16,725980112,3090,50,34,222,43,13,418
3f53b86488768af60e7d5bbb27fe4c553dfe85d8e2e576eb4d1b32c8aef29bb4,2023-07-07,58149378,714,43,25,89,30,13,194
3a083b1e50e6cd7b7bbaf70957cf91256245db6465db1b201cb020c4ec9a4fb3,2023-05-15,95217315,1096,83,60,210,48,11,953
12fb0991d47a426fe038fcf2b8789a625de8f7cd73000939dcc5ecc22fcd3db9,2023-03-17,553634067,2953,44,49,110,66,13,339


In [0]:
# ============================================================
# GOLD — PERSISTÊNCIA DAS TABELAS
# ============================================================

tabelas_gold = {
    "gold_dim_musica": dim_musica,
    "gold_dim_artista": dim_artista,
    "gold_bridge_musica_artista": bridge_musica_artista,
    "gold_dim_tempo": dim_tempo,
    "gold_fato_desempenho_musica": fato_desempenho_musica
}

for nome, dataframe in tabelas_gold.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(nome)
    )
    print(nome, "->", spark.table(nome).count(), "registros")

gold_dim_musica -> 953 registros
gold_dim_artista -> 699 registros
gold_bridge_musica_artista -> 1479 registros
gold_dim_tempo -> 477 registros
gold_fato_desempenho_musica -> 953 registros


In [0]:
# ============================================================
# GOLD — VALIDAÇÃO DE INTEGRIDADE REFERENCIAL
# ============================================================

musica_sem_dim = (
    fato_desempenho_musica
    .join(dim_musica, "musica_id", "left_anti")
    .count()
)

data_sem_dim = (
    fato_desempenho_musica
    .join(dim_tempo, "data_id", "left_anti")
    .count()
)

bridge_sem_musica = (
    bridge_musica_artista
    .join(dim_musica, "musica_id", "left_anti")
    .count()
)

bridge_sem_artista = (
    bridge_musica_artista
    .join(dim_artista, "artista_id", "left_anti")
    .count()
)

print("=== INTEGRIDADE REFERENCIAL GOLD ===")
print("Fato sem música:", musica_sem_dim)
print("Fato sem data:", data_sem_dim)
print("Bridge sem música:", bridge_sem_musica)
print("Bridge sem artista:", bridge_sem_artista)

=== INTEGRIDADE REFERENCIAL GOLD ===
Fato sem música: 0
Fato sem data: 0
Bridge sem música: 0
Bridge sem artista: 0



# Camada Gold — Resultado da Modelagem

Após o tratamento e a validação dos dados na camada Silver, foram construídas as estruturas analíticas da camada Gold.

A modelagem foi organizada para preservar o nível de análise das músicas e, ao mesmo tempo, permitir consultas por artista, período de lançamento e métricas de desempenho.

## Estruturas construídas

### `gold_dim_musica`

A dimensão de músicas contém **953 registros**, correspondentes às 953 identidades analíticas distintas encontradas na base.

Cada música recebeu uma chave técnica `musica_id`, gerada a partir dos atributos descritivos e musicais utilizados na validação da identidade dos registros.

### `gold_dim_artista`

A dimensão de artistas contém **699 artistas distintos**, cada um identificado por uma chave técnica `artista_id`.

### `gold_bridge_musica_artista`

A relação entre músicas e artistas foi modelada separadamente para representar corretamente as colaborações existentes na base.

A bridge contém **1.479 relações música-artista**, abrangendo as **953 músicas** e os **699 artistas** identificados.

Essa estrutura permite analisar individualmente os artistas associados a cada música sem duplicar as métricas de desempenho armazenadas na tabela fato.

### `gold_dim_tempo`

A dimensão de tempo contém **477 datas de lançamento distintas**, sem valores nulos, organizadas pelos atributos de data, ano, mês e dia.

### `gold_fato_desempenho_musica`

A tabela fato contém **953 registros**, mantendo uma observação para cada identidade de música definida no modelo.

Nela estão concentradas as métricas de desempenho disponíveis na base, incluindo streams e presença em playlists e charts das diferentes plataformas.

O único valor inválido de `streams` identificado anteriormente permanece como `NULL`, conforme o tratamento definido na camada Silver.

## Validação da modelagem

Após a persistência das tabelas em formato Delta, foi realizada uma verificação de integridade referencial entre as estruturas da Gold.

Os testes apresentaram os seguintes resultados:

- registros da fato sem correspondência em `gold_dim_musica`: **0**;
- registros da fato sem correspondência em `gold_dim_tempo`: **0**;
- relações da bridge sem correspondência em `gold_dim_musica`: **0**;
- relações da bridge sem correspondência em `gold_dim_artista`: **0**.

Dessa forma, as estruturas da camada Gold estão consistentes entre si e prontas para serem utilizadas nas análises das perguntas definidas para o projeto.


# Análises da Camada Gold

Com a construção e validação da camada Gold, os dados estão organizados para responder às perguntas definidas no início do projeto.

As análises desta etapa utilizam as dimensões, a tabela fato e a relação entre músicas e artistas construídas durante a modelagem. Para cada pergunta, os resultados obtidos serão apresentados e discutidos a partir dos dados disponíveis.

## P1 — Quais músicas e artistas estão associados aos maiores volumes de streams?

A primeira análise busca identificar as músicas com maior número de streams e os artistas associados a essas faixas.

Como uma música pode possuir mais de um artista, a associação entre músicas e artistas será realizada por meio da `gold_bridge_musica_artista`.

O número de streams pertence à música e, por isso, será considerado uma única vez na análise das faixas, independentemente da quantidade de artistas associados. Dessa forma, as colaborações são preservadas sem multiplicar artificialmente o volume de reproduções.

Além da identificação das músicas com maior alcance, será observada a presença dos artistas entre essas faixas para compreender quais nomes aparecem associados aos maiores volumes de streams da base.

In [0]:
# ============================================================
# P1 — MÚSICAS COM MAIORES VOLUMES DE STREAMS
# ============================================================

top_musicas = (
    fato_desempenho_musica
    .join(dim_musica, "musica_id", "inner")
    .join(bridge_musica_artista, "musica_id", "inner")
    .join(dim_artista, "artista_id", "inner")
    .groupBy(
        "musica_id",
        "track_name",
        "streams"
    )
    .agg(
        F.concat_ws(
            ", ",
            F.sort_array(
                F.collect_set("artist_name")
            )
        ).alias("artistas")
    )
    .orderBy(F.desc("streams"))
)

print("=== P1 — TOP 20 MÚSICAS POR STREAMS ===")

display(
    top_musicas
    .select(
        "track_name",
        "artistas",
        "streams"
    )
    .limit(20)
)

=== P1 — TOP 20 MÚSICAS POR STREAMS ===


track_name,artistas,streams
Blinding Lights,The Weeknd,3703895074
Shape of You,Ed Sheeran,3562543890
Someone You Loved,Lewis Capaldi,2887241814
Dance Monkey,Tones and I,2864791672
Sunflower - Spider-Man: Into the Spider-Verse,"Post Malone, Swae Lee",2808096550
One Dance,"Drake, Kyla, WizKid",2713922350
STAY (with Justin Bieber),"Justin Bieber, The Kid Laroi",2665343922
Believer,Imagine Dragons,2594040133
Closer,"Halsey, The Chainsmokers",2591224264
Starboy,"Daft Punk, The Weeknd",2565529693


In [0]:
# ============================================================
# P1 — ARTISTAS ASSOCIADOS AOS MAIORES VOLUMES DE STREAMS
# ============================================================

top_artistas = (
    fato_desempenho_musica
    .join(bridge_musica_artista, "musica_id", "inner")
    .join(dim_artista, "artista_id", "inner")
    .groupBy("artista_id", "artist_name")
    .agg(
        F.sum("streams").alias("streams_associados"),
        F.countDistinct("musica_id").alias("qtd_musicas")
    )
    .orderBy(F.desc("streams_associados"))
)

print("=== P1 — TOP 20 ARTISTAS POR STREAMS ASSOCIADOS ===")

display(
    top_artistas
    .select(
        "artist_name",
        "qtd_musicas",
        "streams_associados"
    )
    .limit(20)
)

=== P1 — TOP 20 ARTISTAS POR STREAMS ASSOCIADOS ===


artist_name,qtd_musicas,streams_associados
The Weeknd,37,23929760757
Bad Bunny,40,23813527270
Ed Sheeran,14,15316587718
Taylor Swift,38,14630378183
Harry Styles,17,11608645649
Eminem,9,10193727260
Dua Lipa,9,9980020481
Justin Bieber,7,8243081039
Drake,19,8043031261
BTS,13,7780428159



# Resultado da P1

Os resultados mostram que **Blinding Lights**, de The Weeknd, apresenta o maior volume de streams da base, com aproximadamente **3,70 bilhões de reproduções**. Em seguida aparecem **Shape of You**, de Ed Sheeran, com cerca de **3,56 bilhões**, e **Someone You Loved**, de Lewis Capaldi, com aproximadamente **2,89 bilhões**.

Quando a análise é realizada por artista, considerando os streams das músicas às quais cada artista está associado, **The Weeknd** aparece com aproximadamente **23,93 bilhões de streams distribuídos por 37 músicas**, seguido de **Bad Bunny**, com cerca de **23,81 bilhões em 40 músicas**. Ed Sheeran, Taylor Swift e Harry Styles também aparecem entre os artistas com maiores volumes associados.

Esses valores não representam uma divisão dos streams entre os participantes de uma colaboração. Quando uma música possui mais de um artista, seu volume de streams permanece associado à música e aparece na trajetória de cada artista relacionado a ela.

Os resultados indicam também que um alto volume acumulado pode ocorrer de formas diferentes. Alguns artistas aparecem associados a um número maior de músicas, enquanto outros alcançam volumes elevados com um conjunto menor de faixas. Por isso, quantidade de músicas e volume de streams são medidas relacionadas, mas não equivalentes.

# P2 — Relação entre presença em playlists e charts e volume de streams

A segunda análise busca verificar se músicas com maior presença em playlists e charts também apresentam maiores volumes de streams.
Para isso, serão comparadas as métricas de streams com os indicadores de presença nas plataformas disponíveis na base, incluindo Spotify, Apple Music, Deezer e Shazam.
A análise será realizada inicialmente por meio da relação entre essas variáveis, permitindo observar se maiores níveis de exposição nas plataformas tendem a acompanhar maiores volumes de reprodução.
Como associação não implica causalidade, os resultados serão interpretados como relações observadas neste conjunto de dados, sem assumir que a presença em playlists ou charts seja, isoladamente, a causa do maior número de streams.

In [0]:
# ============================================================
# P2 — CORRELAÇÃO ENTRE EXPOSIÇÃO E STREAMS
# ============================================================

variaveis_p2 = [
    "in_spotify_playlists",
    "in_spotify_charts",
    "in_apple_playlists",
    "in_apple_charts",
    "in_deezer_playlists",
    "in_deezer_charts",
    "in_shazam_charts"
]

resultados_p2 = []

for variavel in variaveis_p2:
    dados = fato_desempenho_musica.select(
        "streams", variavel
    ).dropna()

    correlacao = dados.stat.corr("streams", variavel)

    resultados_p2.append(
        (variavel, dados.count(), correlacao)
    )

correlacoes_p2 = spark.createDataFrame(
    resultados_p2,
    ["variavel", "registros_validos", "correlacao_streams"]
)

display(
    correlacoes_p2.orderBy(
        F.desc("correlacao_streams")
    )
)

variavel,registros_validos,correlacao_streams
in_spotify_playlists,952,0.7898221599550612
in_apple_playlists,952,0.7720629840820624
in_deezer_playlists,952,0.5981310122534236
in_apple_charts,952,0.32023434128277684
in_spotify_charts,952,0.24582119983366885
in_deezer_charts,952,0.22859797355472555
in_shazam_charts,902,0.024981380996151808


In [0]:
# ============================================================
# P2 — STREAMS POR NÍVEL DE PRESENÇA EM PLAYLISTS
# ============================================================

from pyspark.sql.window import Window

base_p2 = (
    fato_desempenho_musica
    .select("streams", "in_spotify_playlists")
    .dropna()
    .withColumn(
        "faixa_playlist",
        F.ntile(4).over(
            Window.orderBy("in_spotify_playlists")
        )
    )
)

comparacao_p2 = (
    base_p2
    .groupBy("faixa_playlist")
    .agg(
        F.count("*").alias("musicas"),
        F.round(F.avg("in_spotify_playlists"), 0).alias("media_playlists"),
        F.round(F.avg("streams"), 0).alias("media_streams"),
        F.expr("percentile_approx(streams, 0.5)").alias("mediana_streams")
    )
    .orderBy("faixa_playlist")
)

display(comparacao_p2)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


faixa_playlist,musicas,media_playlists,media_streams,mediana_streams
1,238,495.0,1.45204896E8,122763672
2,238,1435.0,2.08833044E8,181328253
3,238,3509.0,4.54234799E8,399686758
4,238,15371.0,1.24827696E9,1159176109


#Resultado da P2
Os resultados indicam uma associação positiva entre a presença das músicas em playlists e o volume de streams. A relação foi mais forte para as playlists do Spotify, com correlação de aproximadamente 0,79, seguida pelas playlists da Apple Music (0,77) e do Deezer (0,60).
Para complementar a análise, as músicas foram divididas em quatro grupos de acordo com sua presença em playlists do Spotify. No grupo com menor presença, a média foi de aproximadamente 145 milhões de streams. Esse valor aumenta para 209 milhões no segundo grupo, 454 milhões no terceiro e aproximadamente 1,25 bilhão de streams no grupo com maior presença em playlists.
A mediana apresentou o mesmo comportamento crescente, passando de aproximadamente 123 milhões para 1,16 bilhão de streams entre o primeiro e o quarto grupo.
As métricas relacionadas a charts apresentaram associações mais fracas. A correlação variou de aproximadamente 0,23 a 0,32 para Spotify, Apple Music e Deezer. No caso do Shazam, a correlação observada foi próxima de zero (0,02), considerando apenas os registros com informação disponível.
Assim, neste conjunto de dados, músicas presentes em mais playlists tendem a apresentar também maiores volumes de streams, especialmente no Spotify e na Apple Music. Para charts, essa relação é consideravelmente mais fraca. Os resultados representam associação entre as variáveis e não permitem concluir que a maior presença em playlists seja a causa do aumento dos streams.

#P3 — Como o desempenho das músicas varia de acordo com o período de lançamento?
A terceira análise busca observar como o volume de streams se distribui entre músicas lançadas em diferentes períodos.
A data de lançamento disponível na camada Gold permite comparar o desempenho das músicas de acordo com o ano em que foram lançadas. Serão considerados tanto o número de músicas presentes na base quanto a média e a mediana de streams em cada período.
Essa comparação exige cuidado porque o conjunto de dados reúne músicas de diferentes épocas que estavam presentes entre as faixas de destaque consideradas pela fonte. Portanto, os resultados não representam todas as músicas lançadas em cada ano nem permitem comparar diretamente o tamanho do mercado musical entre períodos.
A análise procura identificar padrões dentro da própria base, observando se músicas de determinados períodos apresentam diferenças de alcance e se faixas mais antigas continuam aparecendo entre os maiores volumes de streams.

In [0]:
# ============================================================
# P3 — DESEMPENHO POR ANO DE LANÇAMENTO
# ============================================================

desempenho_ano = (
    fato_desempenho_musica
    .join(dim_tempo, "data_id", "inner")
    .groupBy("ano")
    .agg(
        F.count("*").alias("qtd_musicas"),
        F.round(F.avg("streams"), 0).alias("media_streams"),
        F.expr(
            "percentile_approx(streams, 0.5)"
        ).alias("mediana_streams"),
        F.max("streams").alias("maior_streams")
    )
    .orderBy("ano")
)

print("=== P3 — DESEMPENHO POR ANO ===")

display(desempenho_ano)

=== P3 — DESEMPENHO POR ANO ===


ano,qtd_musicas,media_streams,mediana_streams,maior_streams
1930,1,9.0598517E7,90598517,90598517
1942,1,3.95591396E8,395591396,395591396
1946,1,3.89771964E8,389771964,389771964
1950,1,4.73248298E8,473248298,473248298
1952,1,3.95591396E8,395591396,395591396
1957,2,4.59981011E8,178660459,741301563
1958,3,4.36854312E8,295998468,769213520
1959,2,2.86708922E8,127027715,446390129
1963,3,4.37087794E8,404664135,663832097
1968,1,1.145727611E9,1145727611,1145727611


In [0]:
# ============================================================
# P3 — DESEMPENHO POR DÉCADA DE LANÇAMENTO
# ============================================================

desempenho_decada = (
    fato_desempenho_musica
    .join(dim_tempo, "data_id", "inner")
    .withColumn(
        "decada",
        (F.floor(F.col("ano") / 10) * 10).cast("int")
    )
    .groupBy("decada")
    .agg(
        F.count("*").alias("qtd_musicas"),
        F.round(F.avg("streams"), 0).alias("media_streams"),
        F.expr(
            "percentile_approx(streams, 0.5)"
        ).alias("mediana_streams"),
        F.max("streams").alias("maior_streams")
    )
    .orderBy("decada")
)

display(desempenho_decada)

decada,qtd_musicas,media_streams,mediana_streams,maior_streams
1930,1,9.0598517E7,90598517,90598517
1940,2,3.9268168E8,389771964,395591396
1950,9,4.08086944E8,395591396,769213520
1960,4,6.14247748E8,404664135,1145727611
1970,7,1.071526491E9,520034544,2197010679
1980,12,8.61096276E8,704171068,1593270737
1990,14,9.72545174E8,918915401,1755214421
2000,20,1.158798597E9,1217120710,1829992958
2010,151,1.179976673E9,983637508,3703895074
2020,733,3.42050064E8,222612678,2665343922


#Resultado da P3
A distribuição por período de lançamento mostra que a base é fortemente concentrada em músicas recentes. Das 953 observações, 733 correspondem a lançamentos da década de 2020 e 151 à década de 2010. As décadas anteriores possuem quantidade muito menor de músicas e, por isso, suas médias devem ser interpretadas com cautela.
Entre as décadas com maior número de observações, as músicas lançadas nos anos 2010 apresentam média de aproximadamente 1,18 bilhão de streams e mediana de 984 milhões. Para os lançamentos dos anos 2020, a média é de aproximadamente 342 milhões e a mediana de 223 milhões.
Essa diferença não permite concluir que músicas mais antigas tenham necessariamente melhor desempenho. Os lançamentos da década de 2010 tiveram mais tempo para acumular reproduções, enquanto muitas músicas da década de 2020 são mais recentes. Além disso, o conjunto analisado reúne músicas que aparecem em uma base de destaque, e não uma amostra completa de todos os lançamentos de cada período.
Os resultados mostram também que músicas de períodos anteriores continuam alcançando volumes elevados de streams. A maior observação entre as músicas lançadas na década de 2010 supera 3,70 bilhões de streams.
Assim, dentro desta base, o período de lançamento está associado a diferenças de desempenho, mas essa relação deve ser analisada considerando tanto o tempo disponível para acumulação de streams quanto a forte concentração de registros recentes.

#P4 — Características musicais apresentam diferenças entre músicas de maior e menor alcance?
A quarta análise busca verificar se características musicais como danceability, energy e valence apresentam diferenças entre músicas com diferentes níveis de alcance.
Para realizar essa comparação, as músicas serão organizadas em grupos de acordo com seu volume de streams. Em seguida, serão comparados os valores das características musicais entre os grupos de menor e maior alcance.
Além de danceability, energy e valence, a base também disponibiliza atributos como acousticness, instrumentalness, liveness e speechiness. Esses atributos poderão ser utilizados como apoio para verificar se existem outros padrões relevantes nos dados.
A análise será interpretada como uma comparação entre grupos dentro deste conjunto de dados. Eventuais diferenças observadas não serão tratadas como evidência de que determinada característica musical seja responsável pelo sucesso de uma música.

In [0]:
# P4 — COMPARAÇÃO DOS GRUPOS

base_p4 = (
    fato_desempenho_musica
    .join(dim_musica, "musica_id")
    .dropna(subset=["streams"])
    .withColumn("grupo_streams", F.ntile(4).over(Window.orderBy("streams")))
)

comparacao_p4 = (
    base_p4.groupBy("grupo_streams")
    .agg(
        F.count("*").alias("musicas"),
        F.round(F.avg("streams"), 0).alias("media_streams"),
        F.round(F.avg("danceability_%"), 2).alias("danceability"),
        F.round(F.avg("energy_%"), 2).alias("energy"),
        F.round(F.avg("valence_%"), 2).alias("valence")
    )
    .orderBy("grupo_streams")
)

display(comparacao_p4)

grupo_streams,musicas,media_streams,danceability,energy,valence
1,238,8.4422853E7,67.49,64.71,51.12
2,238,2.08448933E8,68.63,65.25,53.8
3,238,4.39823393E8,67.24,63.32,51.91
4,238,1.323854521E9,64.58,63.82,48.8


#Resultado da P4
A comparação entre os grupos de menor e maior alcance mostrou diferenças relativamente pequenas nas três características musicais analisadas, apesar da grande diferença observada no volume de streams.
A média de danceability passou de 67,49 no grupo com menor volume de streams para 64,58 no grupo de maior alcance. Para energy, os valores foram 64,71 e 63,82, respectivamente. Em valence, a média passou de 51,12 para 48,80.
Ao mesmo tempo, a média de streams aumentou de aproximadamente 84 milhões no primeiro grupo para 1,32 bilhão no quarto grupo.
Dentro desta base, portanto, músicas com maior alcance não apresentam valores substancialmente maiores de danceability, energy ou valence. As diferenças observadas são pequenas e não revelam, isoladamente, um perfil musical claramente associado aos maiores volumes de streams.
Esse resultado também reforça que o desempenho de uma música não pode ser explicado apenas por essas características. Outros fatores, incluindo exposição em playlists, período de lançamento e aspectos não disponíveis no conjunto de dados, podem estar relacionados ao alcance observado.

#P5 — O consumo está concentrado em uma pequena parcela das músicas e artistas da base?
A quinta análise busca verificar se o volume de streams está distribuído de maneira relativamente uniforme ou se uma parcela menor das músicas concentra uma parte relevante do consumo observado.
Para as músicas, será calculada a participação acumulada no total de streams após a ordenação das faixas da maior para a menor quantidade de reproduções.
A análise por artistas será realizada separadamente, utilizando a relação entre músicas e artistas construída na camada Gold. Como uma música pode possuir mais de um artista, os resultados dessa análise serão interpretados como streams associados às músicas de cada artista, e não como uma divisão individual das reproduções entre os participantes de uma colaboração.
A comparação permitirá observar o grau de concentração existente na base e verificar quanto do volume total está associado às músicas e aos artistas de maior alcance.

In [0]:
# ============================================================
# P5 — CONCENTRAÇÃO DOS STREAMS ENTRE AS MÚSICAS
# ============================================================

base_p5 = (
    fato_desempenho_musica
    .dropna(subset=["streams"])
    .orderBy(F.desc("streams"))
)

total_musicas = base_p5.count()
total_streams = base_p5.agg(F.sum("streams")).first()[0]

print("=== P5 — CONCENTRAÇÃO ENTRE MÚSICAS ===")
print("Músicas válidas:", total_musicas)
print("Total de streams:", total_streams)

for percentual in [0.10, 0.20, 0.50]:
    quantidade = int(total_musicas * percentual)

    streams_top = (
        base_p5
        .limit(quantidade)
        .agg(F.sum("streams"))
        .first()[0]
    )

    participacao = streams_top / total_streams * 100

    print(
        f"Top {int(percentual*100)}% "
        f"({quantidade} músicas): "
        f"{participacao:.2f}% dos streams"
    )

=== P5 — CONCENTRAÇÃO ENTRE MÚSICAS ===
Músicas válidas: 952
Total de streams: 489458828542
Top 10% (95 músicas): 36.65% dos streams
Top 20% (190 músicas): 57.16% dos streams
Top 50% (476 músicas): 85.76% dos streams


In [0]:
# ============================================================
# P5 — CONCENTRAÇÃO DOS STREAMS ASSOCIADOS AOS ARTISTAS
# ============================================================

artistas_p5 = (
    fato_desempenho_musica
    .join(bridge_musica_artista, "musica_id")
    .join(dim_artista, "artista_id")
    .dropna(subset=["streams"])
    .groupBy("artista_id", "artist_name")
    .agg(F.sum("streams").alias("streams_associados"))
    .orderBy(F.desc("streams_associados"))
)

total_artistas = artistas_p5.count()
total_associado = artistas_p5.agg(
    F.sum("streams_associados")
).first()[0]

print("=== P5 — CONCENTRAÇÃO ENTRE ARTISTAS ===")
print("Artistas:", total_artistas)

for p in [0.10, 0.20, 0.50]:
    n = int(total_artistas * p)

    valor = (
        artistas_p5.limit(n)
        .agg(F.sum("streams_associados"))
        .first()[0]
    )

=== P5 — CONCENTRAÇÃO ENTRE ARTISTAS ===
Artistas: 698


# P5 - Resultado da P5
Os resultados mostram que o consumo apresenta concentração tanto entre as músicas quanto entre os artistas da base.
Entre as 952 músicas com informação válida de streams, as 95 músicas que compõem aproximadamente os 10% de maior alcance concentram 36,65% dos streams. Os 20% de maior alcance concentram 57,16%, enquanto metade das músicas responde por 85,76% do volume total de streams.
A concentração também aparece na análise por artistas. Entre os 699 artistas identificados, os 10% com maior volume de streams associados concentram 43,26% desse volume. Os 20% concentram 61,50%, e os 50% de maior alcance chegam a 86,27%.
Na análise por artistas, os valores representam streams associados às músicas de cada artista. Como uma mesma música pode estar relacionada a mais de um artista, principalmente em colaborações, esses valores não devem ser interpretados como uma divisão individual das reproduções nem comparados diretamente ao total de streams utilizado na análise por músicas.
Os resultados indicam, portanto, uma distribuição desigual do alcance dentro da base, com uma parcela menor de músicas e artistas associada a uma parte expressiva do volume observado. A concentração identificada é relevante, embora os dados não apresentem uma distribuição estritamente equivalente à regra 80/20.

#P6 — Associação entre desempenho nas plataformas e recepção dos usuários
A sexta pergunta do projeto buscou avaliar se existe associação entre o desempenho das músicas nas plataformas digitais e as avaliações dos usuários disponíveis em uma fonte musical externa.
Para essa etapa foi avaliada a utilização do MusicBrainz, base aberta de informações musicais que disponibiliza uma API para consulta de gravações e seus respectivos identificadores.
A proposta foi utilizar informações da música e do artista disponíveis na camada Gold para localizar a gravação correspondente no MusicBrainz e, quando disponível, recuperar sua avaliação e quantidade de votos.
A integração somente foi realizada quando houve correspondência suficientemente confiável entre os registros das duas fontes. Não foram criadas associações apenas pela semelhança aproximada dos nomes.
Como teste de viabilidade, foram selecionadas três músicas da base principal: Blinding Lights, de The Weeknd; Shape of You, de Ed Sheeran; e As It Was, de Harry Styles. As três músicas foram corretamente identificadas no MusicBrainz, com score_matching igual a 100, indicando correspondência exata no procedimento utilizado.
Entretanto, a consulta das avaliações mostrou cobertura insuficiente para sustentar a análise proposta. Blinding Lights e Shape of You não possuíam avaliação e apresentavam 0 votos, enquanto As It Was apresentou rating 5 com apenas 1 voto.
Diante dessa baixa disponibilidade de avaliações, a amostra obtida não permite uma comparação quantitativa consistente entre desempenho nas plataformas digitais e avaliação dos usuários.
Por esse motivo, a P6 não foi respondida quantitativamente. A tentativa de enriquecimento foi mantida no projeto por demonstrar a integração técnica com uma fonte externa e, principalmente, por documentar uma limitação encontrada nos dados em vez de produzir uma conclusão que eles não sustentam.

In [0]:
# ============================================================
# P6 — TESTE DE INTEGRAÇÃO COM MUSICBRAINZ
# ============================================================

import requests
import time

musicas_teste = [
    ("Blinding Lights", "The Weeknd"),
    ("Shape of You", "Ed Sheeran"),
    ("As It Was", "Harry Styles")
]

resultados_mb = []

headers = {
    "User-Agent": "PUCRJ-MVP-EngenhariaDados/1.0 (estudo academico)"
}

for musica, artista in musicas_teste:

    url = "https://musicbrainz.org/ws/2/recording/"

    params = {
        "query": f'recording:"{musica}" AND artist:"{artista}"',
        "fmt": "json",
        "limit": 3
    }

    resposta = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    dados = resposta.json()
    gravacoes = dados.get("recordings", [])

    if gravacoes:
        melhor = gravacoes[0]

        resultados_mb.append({
            "musica": musica,
            "artista": artista,
            "mbid": melhor.get("id"),
            "titulo_encontrado": melhor.get("title"),
            "score_matching": melhor.get("score")
        })
    else:
        resultados_mb.append({
            "musica": musica,
            "artista": artista,
            "mbid": None,
            "titulo_encontrado": None,
            "score_matching": None
        })

    time.sleep(1.1)

display(spark.createDataFrame(resultados_mb))

artista,mbid,musica,score_matching,titulo_encontrado
The Weeknd,5bdbfe86-4ad5-4c19-8d29-ee5f30ba628b,Blinding Lights,100,Blinding Lights
Ed Sheeran,c664a576-f0e5-4725-8d52-05ecfe4822d8,Shape of You,100,Shape of You
Harry Styles,53969964-673a-4407-9396-3087be9245f6,As It Was,100,As It Was


In [0]:
# P6 — TESTE DE RATINGS NO MUSICBRAINZ

ratings_teste = []

for item in resultados_mb:

    mbid = item["mbid"]

    url = f"https://musicbrainz.org/ws/2/recording/{mbid}"

    params = {
        "inc": "ratings",
        "fmt": "json"
    }

    resposta = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    dados = resposta.json()
    rating = dados.get("rating", {})

    ratings_teste.append({
        "musica": item["musica"],
        "artista": item["artista"],
        "mbid": mbid,
        "rating": rating.get("value"),
        "votos": rating.get("votes-count")
    })

    time.sleep(1.1)

display(spark.createDataFrame(ratings_teste))

---------------------------------------------------------------------------
PySparkTypeError                          Traceback (most recent call last)
File <command-6776301977106528>, line 36
     26     ratings_teste.append({
     27         "musica": item["musica"],
     28         "artista": item["artista"],
   (...)     31         "votos": rating.get("votes-count")
     32     })
     34     time.sleep(1.1)
---> 36 display(spark.createDataFrame(ratings_teste))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:812, in SparkSession.createDataFrame(self, data, schema, samplingRatio, verifySchema)
    810         _schema = StructType().add("value", _schema)
    811 else:
--> 812     _schema = self._inferSchemaFromList(_data, _cols, configs)
    814     if _cols is not None and cast(int, _num_cols) < len(_cols):
    815         _num_cols = len(_cols)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:527, in SparkSess


### Resultado da P6

O teste de integração com o MusicBrainz mostrou que a identificação das músicas entre as duas fontes é tecnicamente possível. As três músicas utilizadas no teste — **Blinding Lights**, de The Weeknd, **Shape of You**, de Ed Sheeran, e **As It Was**, de Harry Styles — foram localizadas com `score_matching` igual a **100** e receberam identificadores MBID.

Entretanto, a consulta das avaliações mostrou cobertura insuficiente para sustentar a análise proposta. **Blinding Lights** apresentou rating 2 com apenas **1 voto**, **Shape of You** não possuía avaliação e apresentava **0 votos**, enquanto **As It Was** apresentou rating 5 com apenas **1 voto**.

Diante desse resultado, a coleta não foi ampliada para as demais músicas da base. Embora o relacionamento técnico entre as fontes tenha funcionado no teste realizado, a quantidade de avaliações disponível não oferece evidência suficiente para representar de forma confiável a recepção dos usuários.

Por esse motivo, a sexta pergunta não foi respondida quantitativamente. A tentativa de enriquecimento foi mantida como parte do pipeline para registrar tanto a viabilidade técnica da integração quanto a limitação encontrada na fonte externa, evitando produzir uma associação entre popularidade e avaliação baseada em dados insuficientes.

#Catálogo de Dados
Após a construção das camadas Bronze, Silver e Gold, foi elaborado um catálogo das estruturas utilizadas no pipeline.
O objetivo desta etapa é documentar os conjuntos de dados efetivamente construídos, seus atributos e respectivos tipos, facilitando a compreensão da transformação dos dados desde a ingestão até a disponibilização para análise.
O catálogo considera as tabelas spotify_bronze e spotify_silver, além das cinco estruturas da camada Gold: gold_dim_musica, gold_dim_artista, gold_bridge_musica_artista, gold_dim_tempo e gold_fato_desempenho_musica.
Os schemas serão obtidos diretamente das tabelas persistidas no Databricks, de forma que a documentação reflita a estrutura efetivamente implementada no pipeline.

In [0]:
# ============================================================
# CATÁLOGO DE DADOS — ESTRUTURAS FINAIS
# ============================================================

tabelas_catalogo = [
    "spotify_bronze",
    "spotify_silver",
    "gold_dim_musica",
    "gold_dim_artista",
    "gold_bridge_musica_artista",
    "gold_dim_tempo",
    "gold_fato_desempenho_musica"
]

for tabela in tabelas_catalogo:
    print("\n========================================")
    print("TABELA:", tabela)
    print("========================================")

    spark.table(tabela).printSchema()


TABELA: spotify_bronze
root
 |-- track_name: string (nullable = true)
 |-- artist(s)_name: string (nullable = true)
 |-- artist_count: long (nullable = true)
 |-- released_year: long (nullable = true)
 |-- released_month: long (nullable = true)
 |-- released_day: long (nullable = true)
 |-- in_spotify_playlists: long (nullable = true)
 |-- in_spotify_charts: long (nullable = true)
 |-- streams: string (nullable = true)
 |-- in_apple_playlists: long (nullable = true)
 |-- in_apple_charts: long (nullable = true)
 |-- in_deezer_playlists: string (nullable = true)
 |-- in_deezer_charts: long (nullable = true)
 |-- in_shazam_charts: string (nullable = true)
 |-- bpm: long (nullable = true)
 |-- key: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- danceability_%: long (nullable = true)
 |-- valence_%: long (nullable = true)
 |-- energy_%: long (nullable = true)
 |-- acousticness_%: long (nullable = true)
 |-- instrumentalness_%: long (nullable = true)
 |-- liveness_%: long

In [0]:
# ============================================================
# CATÁLOGO DE DADOS — ESTRUTURA CONSOLIDADA
# ============================================================

catalogo = []

for tabela in tabelas_catalogo:
    camada = (
        "Bronze" if tabela == "spotify_bronze"
        else "Silver" if tabela == "spotify_silver"
        else "Gold"
    )

    for campo in spark.table(tabela).schema.fields:
        catalogo.append((
            camada,
            tabela,
            campo.name,
            campo.dataType.simpleString(),
            campo.nullable
        ))

df_catalogo = spark.createDataFrame(
    catalogo,
    ["camada", "tabela", "atributo", "tipo", "aceita_nulo"]
)

display(df_catalogo)

camada,tabela,atributo,tipo,aceita_nulo
Bronze,spotify_bronze,track_name,string,true
Bronze,spotify_bronze,artist(s)_name,string,true
Bronze,spotify_bronze,artist_count,bigint,true
Bronze,spotify_bronze,released_year,bigint,true
Bronze,spotify_bronze,released_month,bigint,true
Bronze,spotify_bronze,released_day,bigint,true
Bronze,spotify_bronze,in_spotify_playlists,bigint,true
Bronze,spotify_bronze,in_spotify_charts,bigint,true
Bronze,spotify_bronze,streams,string,true
Bronze,spotify_bronze,in_apple_playlists,bigint,true


##Autoavaliação e Limitações

O pipeline desenvolvido permitiu percorrer as principais etapas previstas para o MVP: ingestão dos dados, persistência da camada Bronze, análise de qualidade, preparação da Silver, modelagem dimensional da Gold e utilização das estruturas resultantes para responder às perguntas analíticas.

As cinco primeiras perguntas puderam ser respondidas com os dados disponíveis. A sexta pergunta previa o enriquecimento da base com informações externas sobre recepção musical e foi mantida até o final do projeto para avaliar sua viabilidade.

O teste de integração com o MusicBrainz mostrou que a correspondência entre música e artista pode ser realizada tecnicamente. Nas três músicas utilizadas para validação, os registros foram encontrados com `score_matching` igual a **100**. Entretanto, a cobertura das avaliações foi insuficiente: **Blinding Lights** e **Shape of You** não possuíam avaliação e apresentavam **0 votos**, enquanto **As It Was** apresentou **rating 5 com apenas 1 voto**. Por esse motivo, a coleta não foi ampliada para toda a base e a P6 não foi respondida quantitativamente.

Também foram identificadas limitações na própria fonte principal. O dataset utilizado foi disponibilizado no Kaggle e sua documentação informa a utilização de múltiplas fontes, sem detalhar individualmente todas elas. Por esse motivo, o trabalho preservou a proveniência declarada pelo responsável pelo conjunto de dados, sem assumir extração direta da API oficial do Spotify.

Outro ponto relevante é que a base reúne músicas de destaque e não representa o universo completo de lançamentos musicais. Dessa forma, resultados relacionados a período de lançamento, concentração de streams ou características musicais devem ser interpretados dentro dos limites deste conjunto de dados.

Durante a preparação também foram encontrados problemas pontuais de qualidade. Um registro apresentou conteúdo incompatível no campo `streams`, que foi preservado com valor `NULL`, e os valores ausentes de `key` e `in_shazam_charts` foram mantidos quando não havia evidência suficiente para justificar uma imputação.

Como evolução do projeto, seria possível incorporar uma fonte externa com maior cobertura de avaliações, ampliar o período e o universo de músicas analisadas e automatizar verificações adicionais de qualidade e atualização dos dados.

##Conclusão
Este MVP teve como objetivo construir um pipeline de Engenharia de Dados capaz de transformar dados brutos sobre desempenho musical em informações estruturadas e adequadas para análise.
A arquitetura foi organizada nas camadas Bronze, Silver e Gold. A Bronze preservou os dados recebidos e os metadados de ingestão. Na Silver foram realizados os tratamentos de qualidade, padronização e conversão dos dados. Na Gold foi construída uma modelagem dimensional composta por dimensões de músicas, artistas e tempo, uma tabela fato de desempenho e uma bridge para representar a relação muitos-para-muitos entre músicas e artistas.
As análises realizadas mostraram que os maiores volumes de streams estão associados a um conjunto relativamente concentrado de músicas e artistas. Também foi observada uma associação positiva entre presença em playlists e volume de streams, mais evidente do que a relação encontrada com presença em charts.
A comparação entre períodos de lançamento mostrou diferenças de desempenho, mas também evidenciou a necessidade de considerar a concentração de músicas recentes na base e o tempo disponível para acumulação de streams. Já as características danceability, energy e valence apresentaram diferenças relativamente pequenas entre os grupos de menor e maior alcance, não indicando isoladamente um perfil musical claramente associado aos maiores volumes de reprodução.
A tentativa de enriquecimento com uma fonte externa também fez parte do processo. Embora o MusicBrainz tenha permitido identificar corretamente as músicas utilizadas no teste, a baixa quantidade de avaliações disponíveis não ofereceu evidência suficiente para responder quantitativamente à sexta pergunta. A decisão foi manter essa limitação documentada, em vez de ampliar a coleta ou produzir conclusões com dados insuficientes.
Ao final, o pipeline permitiu preservar a rastreabilidade desde a origem dos dados até as estruturas analíticas e demonstrou como decisões de qualidade, modelagem e integração influenciam diretamente aquilo que pode ser concluído a partir dos dados.